In [14]:
# ==========================================================
# CELL 1: ENTERPRISE DEPENDENCIES
# ==========================================================
!pip install -qU langchain-google-genai langchain-community langchain-huggingface
!pip install -qU chromadb pypdf rank_bm25 gradio FlagEmbedding
!pip install -qU sentence-transformers pytesseract pdf2image
!pip install -qU reportlab psutil

# OCR system deps
!apt-get update -qq
!apt-get install -y tesseract-ocr poppler-utils

print("✅ All dependencies installed")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
poppler-utils is already the newest version (22.02.0-2ubuntu0.13).
0 upgraded, 0 newly installed, 0 to remove and 118 not upgraded.
✅ All dependencies installed


In [15]:
# ==========================================================
# CELL 2: OCR SYSTEM DEPENDENCIES
# ==========================================================
!apt-get update -qq
!apt-get install -y tesseract-ocr
!apt-get install -y poppler-utils

print("✅ OCR system dependencies installed.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 118 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.13).
0 upgraded, 0 newly installed, 0 to remove and 118 not upgraded.
✅ OCR system dependencies installed.


In [16]:
# ==========================================================
# CELL 2: ENTERPRISE DOCUMENT INTELLIGENCE PLATFORM v3.3
# Fixes: persistent embedding cache, fast startup, compact BM25,
#        hierarchical reports, query cache, async retrieval
# ==========================================================

import os
import math
import time
import json
import shutil
import hashlib
import pickle
import re
import asyncio
from typing import List, Dict, Tuple, Optional, Set
from collections import defaultdict
from datetime import datetime, timedelta

# LangChain
from langchain_community.document_loaders import (
    PyPDFLoader, Docx2txtLoader, TextLoader, CSVLoader,
    UnstructuredExcelLoader, UnstructuredPowerPointLoader
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document

# ML
from sentence_transformers import CrossEncoder
import numpy as np

# OCR
from PIL import Image
import pytesseract
from pdf2image import convert_from_path

# WordNet for query expansion
import nltk
try:
    nltk.download('wordnet', quiet=True)
    nltk.download('averaged_perceptron_tagger', quiet=True)
    nltk.download('punkt', quiet=True)
    nltk.download('omw-1.4', quiet=True)
except:
    pass
from nltk.corpus import wordnet

# API Key
if "GOOGLE_API_KEY" not in os.environ:
    from google.colab import userdata
    try:
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except:
        import getpass
        os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter Gemini API Key: ")


# ==================== FIX 3: COMPACT INCREMENTAL BM25 ====================

class IncrementalBM25:
    """Compact incremental BM25 storing statistics only, not raw tokens."""

    def __init__(self, cache_file: str = "./bm25_state.pkl"):
        self.cache_file = cache_file
        self.doc_term_freqs: List[Dict[str, int]] = []  # Per-doc term frequencies only
        self.doc_lengths: List[int] = []
        self.doc_freqs: Dict[str, int] = defaultdict(int)
        self.corpus_size: int = 0
        self.avg_doc_len: float = 0.0
        self.k1: float = 1.5
        self.b: float = 0.75
        self._doc_len_sum: int = 0
        self._checksum: str = ""

        self._load_state()
        self._validate_or_rebuild()

    def _tokenize(self, text: str) -> List[str]:
        return re.findall(r'\b[a-zA-Z]{2,}\b', text.lower())

    def _compute_checksum(self) -> str:
        state = str(self.corpus_size) + str(self._doc_len_sum) + str(len(self.doc_term_freqs))
        return hashlib.md5(state.encode()).hexdigest()

    def _load_state(self):
        if os.path.exists(self.cache_file):
            try:
                with open(self.cache_file, 'rb') as f:
                    state = pickle.load(f)
                self.doc_term_freqs = state.get('doc_term_freqs', [])
                self.doc_lengths = state.get('doc_lengths', [])
                self.doc_freqs = defaultdict(int, state.get('doc_freqs', {}))
                self.corpus_size = state.get('corpus_size', 0)
                self._doc_len_sum = state.get('doc_len_sum', 0)
                self.avg_doc_len = state.get('avg_doc_len', 0.0)
                self._checksum = state.get('checksum', '')
                print(f"💾 Loaded BM25 state: {self.corpus_size} docs, {len(self.doc_freqs)} unique terms")
            except Exception as e:
                print(f"⚠️ BM25 load failed: {e}")
                self._reset_state()

    def _reset_state(self):
        self.doc_term_freqs = []
        self.doc_lengths = []
        self.doc_freqs = defaultdict(int)
        self.corpus_size = 0
        self._doc_len_sum = 0
        self.avg_doc_len = 0.0
        self._checksum = ""

    def save_state(self):
        try:
            self._checksum = self._compute_checksum()
            state = {
                'doc_term_freqs': self.doc_term_freqs,
                'doc_lengths': self.doc_lengths,
                'doc_freqs': dict(self.doc_freqs),
                'corpus_size': self.corpus_size,
                'doc_len_sum': self._doc_len_sum,
                'avg_doc_len': self.avg_doc_len,
                'checksum': self._checksum,
                'version': '3.3',
                'timestamp': datetime.now().isoformat()
            }
            with open(self.cache_file, 'wb') as f:
                pickle.dump(state, f)
            print(f"💾 BM25 state saved: {self.corpus_size} docs, {len(self.doc_freqs)} terms")
        except Exception as e:
            print(f"❌ BM25 save failed: {e}")

    def validate_state(self) -> bool:
        if self.corpus_size != len(self.doc_term_freqs):
            print("❌ BM25 validation: corpus_size mismatch")
            return False
        if self.corpus_size != len(self.doc_lengths):
            print("❌ BM25 validation: doc_lengths mismatch")
            return False
        expected_len = sum(self.doc_lengths) if self.doc_lengths else 0
        if self._doc_len_sum != expected_len:
            print("❌ BM25 validation: doc_len_sum mismatch")
            return False
        if self._checksum and self._checksum != self._compute_checksum():
            print("❌ BM25 validation: checksum mismatch")
            return False
        return True

    def _validate_or_rebuild(self):
        if not self.validate_state():
            print("🔧 BM25 state corrupted, resetting...")
            self._reset_state()

    def add_documents(self, texts: List[str]):
        for text in texts:
            tokens = self._tokenize(text)
            if not tokens:
                continue
            # Store term frequencies (compact) instead of raw tokens
            term_freqs = {}
            for token in tokens:
                term_freqs[token] = term_freqs.get(token, 0) + 1

            unique_tokens = set(tokens)
            for token in unique_tokens:
                self.doc_freqs[token] += 1

            self.doc_term_freqs.append(term_freqs)
            self.doc_lengths.append(len(tokens))
            self.corpus_size += 1
            self._doc_len_sum += len(tokens)

        if self.corpus_size > 0:
            self.avg_doc_len = self._doc_len_sum / self.corpus_size
        self.save_state()

    def delete_documents(self, indices: List[int]) -> List[int]:
        indices = sorted(set(indices), reverse=True)
        removed = []
        for idx in indices:
            if 0 <= idx < len(self.doc_term_freqs):
                term_freqs = self.doc_term_freqs.pop(idx)
                doc_len = self.doc_lengths.pop(idx)
                removed.append(idx)
                self.corpus_size -= 1
                self._doc_len_sum -= doc_len
                # Decrement doc frequencies
                for token in term_freqs:
                    self.doc_freqs[token] -= 1
                    if self.doc_freqs[token] <= 0:
                        del self.doc_freqs[token]

        if not removed:
            return []

        if self.corpus_size > 0:
            self.avg_doc_len = self._doc_len_sum / self.corpus_size
        else:
            self.avg_doc_len = 0.0
        self.save_state()
        return removed

    def score(self, query: str, doc_indices: Optional[List[int]] = None) -> List[Tuple[int, float]]:
        query_tokens = self._tokenize(query)
        if not query_tokens:
            return []
        if doc_indices is None:
            doc_indices = range(len(self.doc_term_freqs))

        scores = []
        for idx in doc_indices:
            if idx >= len(self.doc_term_freqs):
                continue
            term_freqs = self.doc_term_freqs[idx]
            doc_len = self.doc_lengths[idx]
            score = 0.0

            for token in query_tokens:
                df = self.doc_freqs.get(token, 0)
                if df == 0:
                    continue
                idf = math.log((self.corpus_size - df + 0.5) / (df + 0.5) + 1.0)
                tf = term_freqs.get(token, 0)
                denom = tf + self.k1 * (1 - self.b + self.b * (doc_len / self.avg_doc_len)) if self.avg_doc_len > 0 else tf + self.k1
                score += idf * (tf * (self.k1 + 1)) / denom if denom > 0 else 0

            scores.append((idx, score))

        scores.sort(key=lambda x: x[1], reverse=True)
        return scores

    def get_top_k(self, query: str, k: int = 5) -> List[int]:
        scores = self.score(query)
        return [idx for idx, _ in scores[:k]]


# ==================== FIX 1: PERSISTENT EMBEDDING CACHE ====================

class PersistentEmbeddingCache:
    """Disk-backed embedding cache with LRU eviction."""

    def __init__(self, cache_file: str = "./embedding_cache.pkl", max_ram_entries: int = 5000):
        self.cache_file = cache_file
        self.max_ram_entries = max_ram_entries
        self._ram_cache: Dict[str, np.ndarray] = {}
        self._access_order: List[str] = []  # LRU tracking
        self._load_from_disk()

    def _load_from_disk(self):
        if os.path.exists(self.cache_file):
            try:
                with open(self.cache_file, 'rb') as f:
                    data = pickle.load(f)
                # Load only metadata, not full vectors (lazy load)
                self._disk_index = {k: True for k in data.keys()}
                print(f"💾 Embedding cache index: {len(self._disk_index)} entries on disk")
            except Exception as e:
                print(f"⚠️ Embedding cache load failed: {e}")
                self._disk_index = {}
        else:
            self._disk_index = {}

    def _save_to_disk(self):
        """Persist current RAM cache to disk."""
        try:
            # Merge with existing disk data
            all_data = {}
            if os.path.exists(self.cache_file):
                with open(self.cache_file, 'rb') as f:
                    all_data = pickle.load(f)
            # Update with RAM cache
            for k, v in self._ram_cache.items():
                all_data[k] = v.tolist()
            with open(self.cache_file, 'wb') as f:
                pickle.dump(all_data, f)
        except Exception as e:
            print(f"❌ Embedding cache save failed: {e}")

    def _evict_lru(self):
        """Evict least recently used entries if over limit."""
        while len(self._ram_cache) > self.max_ram_entries and self._access_order:
            oldest = self._access_order.pop(0)
            if oldest in self._ram_cache:
                del self._ram_cache[oldest]

    def get(self, key: str) -> Optional[np.ndarray]:
        # Check RAM first
        if key in self._ram_cache:
            # Move to end (most recent)
            if key in self._access_order:
                self._access_order.remove(key)
            self._access_order.append(key)
            return self._ram_cache[key]

        # Check disk
        if key in self._disk_index:
            try:
                with open(self.cache_file, 'rb') as f:
                    data = pickle.load(f)
                if key in data:
                    vec = np.array(data[key])
                    self._ram_cache[key] = vec
                    self._access_order.append(key)
                    self._evict_lru()
                    return vec
            except Exception:
                pass

        return None

    def put(self, key: str, vector: np.ndarray):
        self._ram_cache[key] = vector
        if key in self._access_order:
            self._access_order.remove(key)
        self._access_order.append(key)
        self._disk_index[key] = True
        self._evict_lru()

    def delete(self, key: str):
        if key in self._ram_cache:
            del self._ram_cache[key]
        if key in self._access_order:
            self._access_order.remove(key)
        if key in self._disk_index:
            del self._disk_index[key]

    def save(self):
        self._save_to_disk()

    def clear(self):
        self._ram_cache = {}
        self._access_order = []
        self._disk_index = {}
        if os.path.exists(self.cache_file):
            os.remove(self.cache_file)


# ==================== MAIN ENGINE ====================

class CapstoneRAGEngine:
    """Enterprise Document Intelligence Platform v3.3"""

    def __init__(self):
        print("=" * 70)
        print("🚀 ENTERPRISE DOCUMENT INTELLIGENCE PLATFORM v3.3")
        print("=" * 70)

        print("\n📦 Loading BGE-Large Embeddings...")
        self.embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5")

        print("🧠 Initializing Gemini 2.5 Flash...")
        self.llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.1, max_tokens=4096)

        print("⚖️ Loading CrossEncoder Reranker...")
        self.reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

        self.persist_directory = "./chromadb_store"
        self.collection_name = "enterprise_rag"
        self.registry_file = "./document_registry.json"
        self.bm25_cache_file = "./bm25_state.pkl"
        self.calibration_file = "./confidence_calibration.json"
        self.doc_embeddings_file = "./doc_embeddings.pkl"
        self.chunk_metadata_file = "./chunk_metadata.json"

        self.vector_db = None
        self.bm25_engine: Optional[IncrementalBM25] = None
        self.chunk_metadata: List[Dict] = []
        self.chunk_id_to_index: Dict[str, int] = {}
        self.last_reranker_scores: List[float] = []
        self.last_scored_docs: List[Tuple[Document, float]] = []
        self.document_registry: Dict = {}

        # Confidence calibration
        self.score_history: List[float] = []
        self._load_calibration()

        # FIX 1: Persistent embedding cache (RAM + disk)
        self._embedding_cache = PersistentEmbeddingCache("./embedding_cache.pkl", max_ram_entries=5000)

        # FIX 2: Persistent document embeddings
        self.document_embeddings: Dict[str, np.ndarray] = {}
        self._load_doc_embeddings()

        # FIX 7: Query result cache
        self.query_cache: Dict[str, Dict] = {}
        self.cache_ttl_minutes = 30

        self.prompt_template = ChatPromptTemplate.from_template(
            "CRITICAL INSTRUCTIONS:\n"
            "1. Use ONLY information in the provided context.\n"
            "2. Never invent facts, numbers, or page numbers.\n"
            "3. If unsupported, respond: 'I cannot verify this based on the uploaded document.'\n"
            "4. Cite specific sources when referencing information.\n\n"
            "Context:\n{context}\n\n"
            "Question: {input}\n"
            "Answer:"
        )

        self.load_registry()
        self._attempt_chroma_recovery()
        self._verify_ocr()

        print("\n" + "=" * 70)
        print(f"✅ ENGINE READY | Docs: {len(self.document_registry)} | Metadata: {len(self.chunk_metadata)}")
        print("=" * 70)

    # ==================== FIX 2: FAST STARTUP ====================

    def _save_chunk_metadata(self):
        """Persist chunk metadata directly for fast recovery."""
        try:
            with open(self.chunk_metadata_file, 'w') as f:
                json.dump({
                    'metadata': self.chunk_metadata,
                    'chunk_id_to_index': self.chunk_id_to_index
                }, f)
        except Exception as e:
            print(f"❌ Metadata save failed: {e}")

    def _load_chunk_metadata(self) -> bool:
        """Fast load chunk metadata from disk."""
        if not os.path.exists(self.chunk_metadata_file):
            return False
        try:
            with open(self.chunk_metadata_file, 'r') as f:
                data = json.load(f)
            self.chunk_metadata = data.get('metadata', [])
            self.chunk_id_to_index = {k: int(v) for k, v in data.get('chunk_id_to_index', {}).items()}
            print(f"💾 Fast-loaded {len(self.chunk_metadata)} metadata entries")
            return True
        except Exception as e:
            print(f"⚠️ Fast metadata load failed: {e}")
            return False

    # ==================== DOCUMENT EMBEDDING CACHE ====================

    def _load_doc_embeddings(self):
        if os.path.exists(self.doc_embeddings_file):
            try:
                with open(self.doc_embeddings_file, 'rb') as f:
                    data = pickle.load(f)
                self.document_embeddings = {k: np.array(v) for k, v in data.items()}
                print(f"💾 Loaded {len(self.document_embeddings)} document embeddings")
            except Exception as e:
                print(f"⚠️ Doc embedding load failed: {e}")
                self.document_embeddings = {}

    def _save_doc_embeddings(self):
        try:
            data = {k: v.tolist() for k, v in self.document_embeddings.items()}
            with open(self.doc_embeddings_file, 'wb') as f:
                pickle.dump(data, f)
        except Exception as e:
            print(f"❌ Doc embedding save failed: {e}")

    def _get_document_embeddings(self) -> Dict[str, np.ndarray]:
        """Return cached document embeddings. Compute missing ones if needed."""
        for doc_id in self.document_registry:
            if doc_id not in self.document_embeddings:
                doc_chunk_embs = []
                for meta in self.chunk_metadata:
                    if meta.get("document_id") == doc_id:
                        cid = meta.get("chunk_id")
                        emb = self._embedding_cache.get(cid)
                        if emb is not None:
                            doc_chunk_embs.append(emb)
                if doc_chunk_embs:
                    self.document_embeddings[doc_id] = np.mean(doc_chunk_embs, axis=0)
        return self.document_embeddings

    # ==================== CONFIDENCE CALIBRATION ====================

    def _load_calibration(self):
        if os.path.exists(self.calibration_file):
            try:
                with open(self.calibration_file, 'r') as f:
                    data = json.load(f)
                self.score_history = data.get('scores', [])
                print(f"📊 Loaded {len(self.score_history)} calibration samples")
            except Exception as e:
                print(f"⚠️ Calibration load failed: {e}")
                self.score_history = []

    def _save_calibration(self):
        try:
            with open(self.calibration_file, 'w') as f:
                json.dump({'scores': self.score_history[-2000:]}, f)
        except Exception as e:
            print(f"❌ Calibration save failed: {e}")

    def _update_calibration(self, scores: List[float]):
        if not scores:
            return
        self.score_history.extend(scores)
        self.score_history = self.score_history[-2000:]
        self._save_calibration()

    def calibrate_confidence(self, score: float) -> int:
        if len(self.score_history) < 10:
            if len(self.score_history) >= 2:
                arr = np.array(self.score_history)
                s_min, s_max = float(np.min(arr)), float(np.max(arr))
                if s_max > s_min:
                    normalized = (score - s_min) / (s_max - s_min)
                    return max(0, min(100, int(normalized * 100)))
            return max(0, min(100, int((score + 5) / 10 * 100)))

        arr = np.array(self.score_history)
        percentile = np.sum(arr < score) / len(arr) * 100
        return max(0, min(100, int(percentile)))

    # ==================== PERSISTENCE ====================

    def _safe_persist(self):
        if self.vector_db is None:
            return
        try:
            print("💾 Chroma persisted")
        except Exception as e:
            print(f"⚠️ Persist warning: {e}")

    def force_persist(self):
        self._safe_persist()
        self.save_registry()
        if self.bm25_engine:
            self.bm25_engine.save_state()
        self._save_calibration()
        self._save_doc_embeddings()
        self._save_chunk_metadata()
        self._embedding_cache.save()
        print("✅ Manual checkpoint complete")

    # ==================== MEMORY OPTIMIZATION ====================

    def _build_metadata_only(self, chunks: List[Document]):
        for idx, chunk in enumerate(chunks):
            meta = {
                "chunk_id": chunk.metadata.get("chunk_id"),
                "source_file": chunk.metadata.get("source_file"),
                "document_id": chunk.metadata.get("document_id"),
                "page_number": chunk.metadata.get("page_number", 1),
                "ocr": chunk.metadata.get("ocr", False),
                "ocr_confidence": chunk.metadata.get("ocr_confidence"),
                "bm25_index": len(self.chunk_metadata) + idx
            }
            self.chunk_metadata.append(meta)
            if meta["chunk_id"]:
                self.chunk_id_to_index[meta["chunk_id"]] = meta["bm25_index"]

    def get_chunk_by_id(self, chunk_id: str) -> Optional[Document]:
        if self.vector_db is None:
            return None
        try:
            result = self.vector_db.get(ids=[chunk_id], include=["documents", "metadatas"])
            if result and result["documents"] and result["metadatas"]:
                return Document(
                    page_content=result["documents"][0],
                    metadata=result["metadatas"][0]
                )
        except Exception as e:
            print(f"⚠️ Lazy load failed for {chunk_id}: {e}")
        return None

    def _get_chunks_by_indices(self, indices: List[int]) -> List[Document]:
        docs = []
        for idx in indices:
            if 0 <= idx < len(self.chunk_metadata):
                meta = self.chunk_metadata[idx]
                chunk_id = meta.get("chunk_id")
                if chunk_id:
                    doc = self.get_chunk_by_id(chunk_id)
                    if doc:
                        docs.append(doc)
        return docs

    # ==================== REGISTRY ====================

    def load_registry(self):
        if os.path.exists(self.registry_file):
            try:
                with open(self.registry_file, 'r') as f:
                    self.document_registry = json.load(f)
            except Exception as e:
                print(f"⚠️ Registry load failed: {e}")
                self.document_registry = {}
        else:
            self.document_registry = {}

    def save_registry(self):
        try:
            with open(self.registry_file, 'w') as f:
                json.dump(self.document_registry, f, indent=2)
        except Exception as e:
            print(f"❌ Registry save failed: {e}")

    def document_exists(self, file_path: str) -> bool:
        return os.path.basename(file_path) in self.document_registry

    # ==================== FIX 2: FAST CHROMADB RECOVERY ====================

    def _attempt_chroma_recovery(self):
        if not os.path.exists(self.persist_directory):
            return

        try:
            import chromadb
            client = chromadb.PersistentClient(path=self.persist_directory)

            self.vector_db = Chroma(
                client=client,
                collection_name=self.collection_name,
                embedding_function=self.embeddings
            )

            count = self.vector_db._collection.count()
            if count == 0:
                self.vector_db = None
                return

            # FIX 2: Try fast metadata load first
            if self._load_chunk_metadata():
                # Verify count matches
                if len(self.chunk_metadata) == count:
                    print(f"✅ Fast recovery: {len(self.chunk_metadata)} chunks (no embedding regeneration)")
                    self.bm25_engine = IncrementalBM25(self.bm25_cache_file)
                    # Validate BM25
                    bm25_valid = self.bm25_engine.validate_state()
                    bm25_count_match = self.bm25_engine.corpus_size == len(self.chunk_metadata)
                    if not bm25_valid or not bm25_count_match:
                        print(f"🔧 BM25 rebuild needed...")
                        self._rebuild_bm25_from_chroma()
                    return

            # Fallback: slow recovery with embedding regeneration
            print("🐌 Slow recovery: regenerating metadata from Chroma...")
            all_data = self.vector_db.get(include=["metadatas", "documents"])
            self.chunk_metadata = []
            self.chunk_id_to_index = {}

            for i, (meta, doc_text) in enumerate(zip(all_data.get("metadatas", []), all_data.get("documents", []))):
                meta["bm25_index"] = i
                self.chunk_metadata.append(meta)
                cid = meta.get("chunk_id")
                if cid:
                    self.chunk_id_to_index[cid] = i
                    # Cache embedding
                    try:
                        emb = self.embeddings.embed_query(doc_text)
                        self._embedding_cache.put(cid, np.array(emb))
                    except:
                        pass

            self._save_chunk_metadata()
            self.bm25_engine = IncrementalBM25(self.bm25_cache_file)
            self._rebuild_bm25_from_chroma()

        except Exception as e:
            print(f"❌ Recovery failed: {e}")
            self.vector_db = None
            self.chunk_metadata = []
            self.chunk_id_to_index = {}

    def _rebuild_bm25_from_chroma(self):
        """Rebuild BM25 from Chroma documents."""
        if self.bm25_engine:
            self.bm25_engine._reset_state()
        else:
            self.bm25_engine = IncrementalBM25(self.bm25_cache_file)

        all_docs = self.vector_db.get(include=["documents"])
        texts = all_docs.get("documents", [])
        self.bm25_engine.add_documents(texts)

        for i, meta in enumerate(self.chunk_metadata):
            meta["bm25_index"] = i

        print(f"✅ BM25 rebuilt: {self.bm25_engine.corpus_size} docs")

    # ==================== BM25 DELETE WITH INDEX SYNC ====================

    def _remove_from_bm25(self, doc_name: str):
        if not self.bm25_engine:
            return
        indices_to_remove = []
        for i, meta in enumerate(self.chunk_metadata):
            if meta.get("document_id") == doc_name:
                indices_to_remove.append(i)
        if indices_to_remove:
            self.bm25_engine.delete_documents(indices_to_remove)

    def _resync_indices(self):
        for i, meta in enumerate(self.chunk_metadata):
            meta["bm25_index"] = i
            cid = meta.get("chunk_id")
            if cid:
                self.chunk_id_to_index[cid] = i

    def validate_index_integrity(self) -> bool:
        if self.bm25_engine.corpus_size != len(self.chunk_metadata):
            print(f"❌ Integrity: BM25 count {self.bm25_engine.corpus_size} != metadata {len(self.chunk_metadata)}")
            return False
        for i, meta in enumerate(self.chunk_metadata):
            if meta.get("bm25_index") != i:
                print(f"❌ Integrity: metadata[{i}].bm25_index = {meta.get('bm25_index')}")
                return False
            cid = meta.get("chunk_id")
            if cid and self.chunk_id_to_index.get(cid) != i:
                print(f"❌ Integrity: chunk_id_to_index[{cid}] = {self.chunk_id_to_index.get(cid)} != {i}")
                return False
        if not self.bm25_engine.validate_state():
            return False
        print("✅ Index integrity validated")
        return True

    # ==================== OCR ====================

    def _parse_ocr_confidence(self, confidences_raw: List) -> float:
        valid_confs = []
        for c in confidences_raw:
            try:
                if c is None or c == '':
                    continue
                val = float(str(c).strip())
                if val >= 0:
                    valid_confs.append(val)
            except (ValueError, TypeError):
                continue
        if not valid_confs:
            return 0.0
        avg = sum(valid_confs) / len(valid_confs)
        if avg > 1.0:
            avg = avg / 100.0
        return round(min(avg, 1.0), 2)

    def _verify_ocr(self):
        t = shutil.which("tesseract")
        p = shutil.which("pdfinfo")
        if t and p:
            print("✅ OCR Ready")
        else:
            print("⚠️ OCR deps missing")

    def _ocr_pdf(self, file_path: str) -> List[Document]:
        print(f"🔍 OCR: {os.path.basename(file_path)}")
        images = convert_from_path(file_path)
        docs = []
        for i, img in enumerate(images):
            try:
                data = pytesseract.image_to_data(img, output_type=pytesseract.Output.DICT)
                confidences = data.get('conf', [])
                avg_conf = self._parse_ocr_confidence(confidences)
                text = pytesseract.image_to_string(img)
                if text.strip():
                    docs.append(Document(
                        page_content=text,
                        metadata={
                            "source_file": os.path.basename(file_path),
                            "page": i,
                            "page_number": i + 1,
                            "ocr": True,
                            "ocr_engine": "tesseract",
                            "ocr_confidence": avg_conf,
                            "ocr_timestamp": datetime.now().isoformat()
                        }
                    ))
            except Exception as e:
                print(f"⚠️ OCR failed for page {i+1}: {e}")
                continue
        return docs

    def _ocr_image(self, file_path: str) -> List[Document]:
        try:
            img = Image.open(file_path)
            data = pytesseract.image_to_data(img, output_type=pytesseract.Output.DICT)
            confidences = data.get('conf', [])
            avg_conf = self._parse_ocr_confidence(confidences)
            text = pytesseract.image_to_string(img)
            return [Document(
                page_content=text,
                metadata={
                    "source_file": os.path.basename(file_path),
                    "page": 0,
                    "page_number": 1,
                    "ocr": True,
                    "ocr_engine": "tesseract",
                    "ocr_confidence": avg_conf,
                    "ocr_timestamp": datetime.now().isoformat()
                }
            )]
        except Exception as e:
            print(f"⚠️ Image OCR failed: {e}")
            return []

    # ==================== DOCUMENT LOADING ====================

    def load_document(self, file_path: str) -> List[Document]:
        ext = os.path.splitext(file_path)[1].lower()
        base = os.path.basename(file_path)

        if ext == ".pdf":
            try:
                docs = PyPDFLoader(file_path).load()
                total_chars = sum(len(d.page_content) for d in docs)
                if total_chars < 100:
                    print(f"⚠️ Low text ({total_chars} chars), using OCR...")
                    return self._ocr_pdf(file_path)
                for i, d in enumerate(docs):
                    d.metadata["source_file"] = base
                    d.metadata["page_number"] = d.metadata.get("page", 0) + 1
                    d.metadata["ocr"] = False
                return docs
            except Exception as e:
                print(f"⚠️ PDF load failed: {e}, trying OCR...")
                return self._ocr_pdf(file_path)

        elif ext in [".png", ".jpg", ".jpeg", ".tiff", ".bmp"]:
            return self._ocr_image(file_path)

        elif ext == ".docx":
            loader = Docx2txtLoader(file_path)
        elif ext == ".txt":
            loader = TextLoader(file_path, encoding="utf-8")
        elif ext == ".csv":
            loader = CSVLoader(file_path, encoding="utf-8")
        elif ext == ".md":
            loader = TextLoader(file_path, encoding="utf-8")
        elif ext == ".json":
            with open(file_path, 'r') as f:
                data = json.load(f)
            return [Document(
                page_content=json.dumps(data, indent=2),
                metadata={"source_file": base, "page_number": 1, "ocr": False}
            )]
        elif ext in [".xlsx", ".xls"]:
            loader = UnstructuredExcelLoader(file_path)
        elif ext == ".pptx":
            loader = UnstructuredPowerPointLoader(file_path)
        else:
            raise ValueError(f"Unsupported: {ext}")

        docs = loader.load()
        for d in docs:
            d.metadata["source_file"] = base
            d.metadata.setdefault("page_number", d.metadata.get("page", 0) + 1)
            d.metadata["ocr"] = False
        return docs

    # ==================== INDEXING ====================

    def ingest_and_index(self, file_path: str) -> str:
        base = os.path.basename(file_path)

        if self.document_exists(file_path):
            return f"⚠️ '{base}' already indexed"

        print(f"\n📥 Ingesting: {base}")
        docs = self.load_document(file_path)
        print(f"   📄 {len(docs)} pages loaded")

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000, chunk_overlap=200,
            separators=["\n\n", "\n", " ", ""]
        )
        chunks = splitter.split_documents(docs)
        print(f"   ✂️ {len(chunks)} chunks")

        for idx, chunk in enumerate(chunks):
            chunk.metadata["chunk_id"] = f"{base}_{idx}"
            chunk.metadata["source_file"] = base
            chunk.metadata["document_id"] = base
            chunk.metadata.setdefault("page_number", chunk.metadata.get("page", 0) + 1)

        ids = [c.metadata["chunk_id"] for c in chunks]

        if self.vector_db is None:
            import chromadb
            client = chromadb.PersistentClient(path=self.persist_directory)
            self.vector_db = Chroma(
                client=client,
                collection_name=self.collection_name,
                embedding_function=self.embeddings
            )
            self.vector_db.add_documents(chunks, ids=ids)
        else:
            self.vector_db.add_documents(chunks, ids=ids)

        self._safe_persist()

        # Cache embeddings persistently
        for chunk in chunks:
            cid = chunk.metadata.get("chunk_id")
            if cid:
                try:
                    emb = self.embeddings.embed_query(chunk.page_content)
                    self._embedding_cache.put(cid, np.array(emb))
                except Exception as e:
                    print(f"⚠️ Embedding cache failed for {cid}: {e}")

        # Compute and cache document-level embedding
        doc_chunk_embs = []
        for c in chunks:
            cid = c.metadata.get("chunk_id")
            emb = self._embedding_cache.get(cid)
            if emb is not None:
                doc_chunk_embs.append(emb)
        if doc_chunk_embs:
            self.document_embeddings[base] = np.mean(doc_chunk_embs, axis=0)
            self._save_doc_embeddings()

        self._incremental_bm25_update(chunks)
        self._build_metadata_only(chunks)
        self._save_chunk_metadata()

        unique_pages = len(set(c.metadata.get("page_number", 0) for c in chunks))
        self.document_registry[base] = {
            "document_id": base,
            "chunks_count": len(chunks),
            "pages": unique_pages,
            "upload_timestamp": datetime.now().isoformat(),
            "file_size_mb": round(os.path.getsize(file_path) / (1024*1024), 2)
        }
        self.save_registry()

        return f"✅ '{base}': {len(docs)} pages → {len(chunks)} chunks (Total metadata: {len(self.chunk_metadata)})"

    def _incremental_bm25_update(self, chunks: List[Document]):
        if not chunks:
            return
        if self.bm25_engine is None:
            self.bm25_engine = IncrementalBM25(self.bm25_cache_file)
        texts = [c.page_content for c in chunks]
        self.bm25_engine.add_documents(texts)
        for i, chunk in enumerate(chunks):
            idx = self.bm25_engine.corpus_size - len(chunks) + i
            chunk.metadata["bm25_index"] = idx

    # ==================== QUERY EXPANSION ====================

    def expand_query(self, query: str) -> List[str]:
        expanded = {query.lower()}
        try:
            words = nltk.word_tokenize(query.lower())
            tagged = nltk.pos_tag(words)
        except:
            words = query.lower().split()
            tagged = [(w, 'NN') for w in words]

        for word, tag in tagged:
            if len(expanded) >= 5:
                break
            if not tag.startswith('N'):
                continue
            synonyms = set()
            for syn in wordnet.synsets(word):
                for lemma in syn.lemmas():
                    syn_name = lemma.name().replace('_', ' ')
                    if syn_name != word and syn_name not in synonyms:
                        synonyms.add(syn_name)
                    if len(synonyms) >= 3:
                        break
                if len(synonyms) >= 3:
                    break
            for syn in list(synonyms)[:2]:
                new_query = query.lower().replace(word, syn)
                expanded.add(new_query)
                if len(expanded) >= 5:
                    break

        q_lower = query.lower()
        if "what is" in q_lower:
            expanded.add(q_lower.replace("what is", "tell me about"))
        if "how does" in q_lower:
            expanded.add(q_lower.replace("how does", "explain"))

        return list(expanded)[:5]

    # ==================== DOCUMENT-LEVEL RETRIEVAL ====================

    def _retrieve_documents(self, query: str, top_n: int = 3) -> List[str]:
        if len(self.document_registry) <= top_n:
            return list(self.document_registry.keys())

        query_emb = np.array(self.embeddings.embed_query(query))
        doc_embs = self._get_document_embeddings()

        scores = []
        for doc_id, emb in doc_embs.items():
            norm = np.linalg.norm(query_emb) * np.linalg.norm(emb)
            sim = np.dot(query_emb, emb) / (norm + 1e-8) if norm > 0 else 0
            scores.append((doc_id, float(sim)))

        scores.sort(key=lambda x: x[1], reverse=True)
        return [doc_id for doc_id, _ in scores[:top_n]]

    # ==================== METADATA FILTERS ====================

    def _apply_metadata_filters(self, docs: List[Document], filters: Optional[Dict] = None) -> List[Document]:
        if not filters:
            return docs

        filtered = []
        for doc in docs:
            meta = doc.metadata
            match = True

            if "source_file" in filters and meta.get("source_file") != filters["source_file"]:
                match = False
            if "document_id" in filters and meta.get("document_id") != filters["document_id"]:
                match = False
            if "page_range" in filters:
                start, end = filters["page_range"]
                page = meta.get("page_number", 1)
                if not (start <= page <= end):
                    match = False
            if "ocr_only" in filters and filters["ocr_only"] and not meta.get("ocr", False):
                match = False
            if "uploaded_after" in filters:
                doc_name = meta.get("source_file")
                if doc_name and doc_name in self.document_registry:
                    upload_time_str = self.document_registry[doc_name].get("upload_timestamp", "")
                    try:
                        upload_dt = datetime.fromisoformat(upload_time_str)
                        filter_dt = datetime.fromisoformat(filters["uploaded_after"])
                        if upload_dt < filter_dt:
                            match = False
                    except (ValueError, TypeError):
                        pass

            if match:
                filtered.append(doc)

        return filtered

    # ==================== RETRIEVAL ====================

    def detect_query_type(self, query: str) -> Tuple[str, Optional[str]]:
        q_lower = query.lower()
        doc_filter = None

        for doc_name in self.document_registry.keys():
            base = os.path.splitext(doc_name)[0].lower()
            if doc_name.lower() in q_lower or base in q_lower:
                doc_filter = doc_name
                break

        report_kws = ["complete report", "full report", "generate report", "detailed report"]
        summary_kws = ["summarize", "summary", "overview", "study guide", "high level"]
        page_kws = ["page ", "which page", "what is on page"]

        if any(k in q_lower for k in report_kws):
            return "REPORT", doc_filter
        if any(k in q_lower for k in summary_kws):
            return "SUMMARY", doc_filter
        if any(k in q_lower for k in page_kws):
            return "PAGE_LOOKUP", doc_filter
        return "QA", doc_filter

    def reciprocal_rank_fusion(self, dense: List[Document], sparse: List[Document], k: int = 60) -> List[Document]:
        scores = defaultdict(float)
        docs_map = {}

        for rank, doc in enumerate(dense):
            uid = doc.metadata.get("chunk_id", hash(doc.page_content))
            scores[uid] += 1.0 / (k + rank)
            docs_map[uid] = doc

        for rank, doc in enumerate(sparse):
            uid = doc.metadata.get("chunk_id", hash(doc.page_content))
            scores[uid] += 1.0 / (k + rank)
            docs_map[uid] = doc

        sorted_uids = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return [docs_map[uid] for uid, _ in sorted_uids]

    # ==================== FIX 6: ASYNC RETRIEVAL ====================

    async def _async_dense_retrieve(self, query: str, k: int) -> List[Document]:
        """Async wrapper for dense retrieval."""
        return self.vector_db.similarity_search(query, k=k)

    async def _async_sparse_retrieve(self, query: str, k: int) -> List[Document]:
        """Async wrapper for BM25 retrieval."""
        bm25_indices = self.bm25_engine.get_top_k(query, k=k)
        return self._get_chunks_by_indices(bm25_indices)

    async def _async_retrieve_parallel(self, queries: List[str], dense_k: int, candidate_docs: Optional[List[str]] = None) -> Tuple[List[Document], List[Document]]:
        """Run dense and sparse retrieval in parallel."""
        all_dense_tasks = [self._async_dense_retrieve(q, dense_k) for q in queries]
        all_sparse_tasks = [self._async_sparse_retrieve(q, dense_k) for q in queries]

        dense_results = await asyncio.gather(*all_dense_tasks)
        sparse_results = await asyncio.gather(*all_sparse_tasks)

        all_dense = [d for sublist in dense_results for d in sublist]
        all_sparse = [d for sublist in sparse_results for d in sublist]

        # Apply document filter
        if candidate_docs:
            all_dense = [d for d in all_dense if d.metadata.get("document_id") in candidate_docs]
            all_sparse = [d for d in all_sparse if d.metadata.get("document_id") in candidate_docs]

        return all_dense, all_sparse

    def hybrid_retrieve(self, query: str, filters: Optional[Dict] = None) -> List[Document]:
        if not self.vector_db or not self.bm25_engine:
            return []

        q_type, doc_filter = self.detect_query_type(query)

        dense_k = {"QA": 8, "PAGE_LOOKUP": 10, "SUMMARY": 25, "REPORT": 50}.get(q_type, 8)
        final_k = {"QA": 4, "PAGE_LOOKUP": 6, "SUMMARY": 15, "REPORT": 40}.get(q_type, 4)

        queries = self.expand_query(query)

        # Document-level retrieval
        candidate_docs = None
        if len(self.document_registry) > 5 and q_type == "QA":
            candidate_docs = self._retrieve_documents(query, top_n=3)
            print(f"📚 Document filter: {candidate_docs}")

        # FIX 6: Parallel retrieval
        try:
            all_dense, all_sparse = asyncio.run(self._async_retrieve_parallel(queries, dense_k, candidate_docs))
        except RuntimeError:
            # Fallback to sequential if event loop already running
            all_dense, all_sparse = [], []
            for q in queries:
                all_dense.extend(self.vector_db.similarity_search(q, k=dense_k))
                bm25_indices = self.bm25_engine.get_top_k(q, k=dense_k)
                all_sparse.extend(self._get_chunks_by_indices(bm25_indices))
            if candidate_docs:
                all_dense = [d for d in all_dense if d.metadata.get("document_id") in candidate_docs]
                all_sparse = [d for d in all_sparse if d.metadata.get("document_id") in candidate_docs]

        # Apply metadata filters
        all_dense = self._apply_metadata_filters(all_dense, filters)
        all_sparse = self._apply_metadata_filters(all_sparse, filters)

        # Document name filter
        if doc_filter:
            all_dense = [d for d in all_dense if d.metadata.get("source_file") == doc_filter]
            all_sparse = [d for d in all_sparse if d.metadata.get("source_file") == doc_filter]

        # RRF
        fused = self.reciprocal_rank_fusion(all_dense, all_sparse)

        # Deduplicate
        seen = set()
        unique = []
        for d in fused:
            cid = d.metadata.get("chunk_id", hash(d.page_content))
            if cid not in seen:
                seen.add(cid)
                unique.append(d)

        if q_type == "REPORT":
            return self._report_mode(unique, doc_filter)

        if q_type == "SUMMARY":
            self.last_reranker_scores = [3.0] * min(len(unique), final_k)
            return unique[:final_k]

        # QA: CrossEncoder rerank
        if unique:
            corpus = [d.page_content for d in unique[:20]]
            pairs = [[query, text] for text in corpus]
            scores = self.reranker.predict(pairs)

            scored = list(zip(unique[:20], scores))
            scored.sort(key=lambda x: x[1], reverse=True)

            self.last_reranker_scores = [float(s) for _, s in scored[:final_k]]
            self.last_scored_docs = scored[:final_k]
            self._update_calibration(self.last_reranker_scores)
            return [d for d, _ in scored[:final_k]]

        return []

    def _report_mode(self, docs: List[Document], doc_filter: Optional[str]) -> List[Document]:
        covered = set()
        result = []

        for d in sorted(docs, key=lambda x: (x.metadata.get("source_file", ""),
                                              x.metadata.get("page_number", 0))):
            key = (d.metadata.get("source_file"), d.metadata.get("page_number"))
            if key not in covered:
                result.append(d)
                covered.add(key)

        if doc_filter:
            doc_chunk_ids = [
                m["chunk_id"] for m in self.chunk_metadata
                if m.get("document_id") == doc_filter and m.get("chunk_id")
            ]
            for cid in doc_chunk_ids:
                doc = self.get_chunk_by_id(cid)
                if doc:
                    key = (doc.metadata.get("source_file"), doc.metadata.get("page_number"))
                    if key not in covered:
                        result.append(doc)
                        covered.add(key)

        self.last_reranker_scores = [3.0] * len(result)
        return result

    # ==================== SEMANTIC COMPRESSION ====================

    def semantic_compress_context(self, docs: List[Document], max_chunks: int = 8, similarity_threshold: float = 0.90) -> List[Document]:
        if not docs or len(docs) <= max_chunks:
            return docs

        embeddings = []
        valid_docs = []
        for d in docs:
            cid = d.metadata.get("chunk_id")
            emb = self._embedding_cache.get(cid)
            if emb is not None:
                embeddings.append(emb)
                valid_docs.append(d)

        if len(valid_docs) <= max_chunks:
            return valid_docs

        embeddings = np.array(embeddings)
        norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
        norms[norms == 0] = 1
        embeddings = embeddings / norms

        doc_scores = {}
        for d in valid_docs:
            cid = d.metadata.get("chunk_id")
            for sd, sc in self.last_scored_docs:
                if sd.metadata.get("chunk_id") == cid:
                    doc_scores[cid] = sc
                    break
            if cid not in doc_scores:
                doc_scores[cid] = 0.0

        sorted_indices = sorted(range(len(valid_docs)),
                               key=lambda i: doc_scores.get(valid_docs[i].metadata.get("chunk_id"), 0),
                               reverse=True)

        selected = []
        selected_embeddings = []

        for idx in sorted_indices:
            emb = embeddings[idx]
            too_similar = False
            for sel_emb in selected_embeddings:
                sim = float(np.dot(emb, sel_emb))
                if sim > similarity_threshold:
                    too_similar = True
                    break
            if not too_similar:
                selected.append(valid_docs[idx])
                selected_embeddings.append(emb)
                if len(selected) >= max_chunks:
                    break

        sources = set(d.metadata.get("source_file") for d in selected)
        if len(sources) < 2 and len(self.document_registry) > 1:
            for d in valid_docs:
                if d.metadata.get("source_file") not in sources:
                    selected.append(d)
                    break

        return selected

    def compress_context(self, docs: List[Document], max_chunks: int = 8) -> List[Document]:
        if not docs:
            return []

        page_best = {}
        for doc in docs:
            key = (doc.metadata.get("source_file"), doc.metadata.get("page_number"))
            score = 0
            for sd, sc in self.last_scored_docs:
                if sd.metadata.get("chunk_id") == doc.metadata.get("chunk_id"):
                    score = sc
                    break
            if key not in page_best or score > page_best[key][1]:
                page_best[key] = (doc, score)

        deduped = [doc for doc, _ in sorted(page_best.values(), key=lambda x: x[1], reverse=True)]
        return self.semantic_compress_context(deduped, max_chunks=max_chunks)

    def retrieval_confidence(self) -> int:
        if not self.last_reranker_scores:
            return 0
        max_score = max(self.last_reranker_scores)
        return self.calibrate_confidence(max_score)

    # ==================== FAST CITATION SPANS ====================

    def _extract_citation_spans(self, answer: str, docs: List[Document]) -> List[Dict]:
        spans = []
        answer_sentences = re.split(r'(?<=[.!?])\s+', answer)

        for doc in docs:
            doc_text = doc.page_content
            doc_sentences = re.split(r'(?<=[.!?])\s+', doc_text)
            doc_spans = []

            for sent in answer_sentences:
                sent_clean = sent.strip()
                if len(sent_clean) < 10:
                    continue
                sent_lower = sent_clean.lower()
                sent_words = set(sent_lower.split())
                if not sent_words:
                    continue

                best_match = None
                best_score = 0.0

                for ds in doc_sentences:
                    ds_stripped = ds.strip()
                    if sent_lower in ds_stripped.lower():
                        best_match = ds_stripped
                        best_score = 1.0
                        break

                if not best_match:
                    for ds in doc_sentences:
                        ds_words = set(ds.lower().split())
                        overlap = len(sent_words & ds_words) / len(sent_words) if sent_words else 0
                        if overlap > best_score:
                            best_score = overlap
                            best_match = ds.strip()

                if best_match and best_score >= 0.5:
                    doc_spans.append({
                        "sentence": sent_clean,
                        "evidence": best_match,
                        "score": round(best_score, 3)
                    })

            if doc_spans:
                spans.append({
                    "source_file": doc.metadata.get("source_file"),
                    "page": doc.metadata.get("page_number"),
                    "chunk_id": doc.metadata.get("chunk_id"),
                    "spans": doc_spans
                })

        return spans

    # ==================== FIX 4: HIERARCHICAL REPORT GENERATION ====================

    def _generate_page_summaries(self, docs: List[Document]) -> Dict[Tuple[str, int], str]:
        """Generate summaries for each unique page."""
        page_groups = defaultdict(list)
        for d in docs:
            key = (d.metadata.get("source_file"), d.metadata.get("page_number"))
            page_groups[key].append(d.page_content)

        summaries = {}
        for (source, page), contents in page_groups.items():
            combined = "\n".join(contents)[:2000]  # Limit per page
            prompt = f"Summarize this page in 2-3 sentences:\n\n{combined}\n\nSummary:"
            try:
                response = self.llm.invoke(prompt)
                summary = response.content if hasattr(response, 'content') else str(response)
                summaries[(source, page)] = summary.strip()
            except Exception:
                summaries[(source, page)] = combined[:300]

        return summaries

    def _generate_document_summary(self, page_summaries: Dict[Tuple[str, int], str], doc_name: str) -> str:
        """Aggregate page summaries into document summary."""
        doc_pages = {k: v for k, v in page_summaries.items() if k[0] == doc_name}
        if not doc_pages:
            return ""

        combined = "\n\n".join([f"Page {p}: {s}" for (_, p), s in sorted(doc_pages.items())])
        prompt = f"Synthesize these page summaries into a coherent document summary:\n\n{combined}\n\nDocument Summary:"
        try:
            response = self.llm.invoke(prompt)
            return response.content if hasattr(response, 'content') else str(response)
        except Exception:
            return combined[:500]

    def generate_hierarchical_report(self, query: str) -> str:
        """Generate report using hierarchical summarization."""
        # Retrieve all relevant chunks
        retrieved = self.hybrid_retrieve(query)
        if not retrieved:
            return "I cannot verify this based on the uploaded document."

        # Stage 1: Page summaries
        print("📝 Generating page summaries...")
        page_summaries = self._generate_page_summaries(retrieved)

        # Stage 2: Document summaries
        print("📄 Generating document summaries...")
        doc_names = set(d.metadata.get("source_file") for d in retrieved)
        doc_summaries = {}
        for doc_name in doc_names:
            doc_summaries[doc_name] = self._generate_document_summary(page_summaries, doc_name)

        # Stage 3: Final report synthesis
        print("📊 Synthesizing final report...")
        report_parts = []
        for doc_name, summary in doc_summaries.items():
            report_parts.append(f"## {doc_name}\n\n{summary}")

        combined = "\n\n".join(report_parts)
        prompt = f"Generate a comprehensive report from these document summaries:\n\n{combined}\n\nFinal Report:"

        try:
            response = self.llm.invoke(prompt)
            return response.content if hasattr(response, 'content') else str(response)
        except Exception as e:
            return f"Report generation failed: {e}"

    # ==================== FIX 7: QUERY CACHE ====================

    def _get_cache_key(self, query: str, filters: Optional[Dict] = None) -> str:
        """Generate cache key from query and filters."""
        key_data = query.lower().strip()
        if filters:
            key_data += "|" + json.dumps(filters, sort_keys=True)
        return hashlib.md5(key_data.encode()).hexdigest()

    def _get_cached_result(self, query: str, filters: Optional[Dict] = None) -> Optional[Tuple]:
        """Check query cache for valid result."""
        cache_key = self._get_cache_key(query, filters)
        if cache_key in self.query_cache:
            entry = self.query_cache[cache_key]
            # Check TTL
            cached_time = datetime.fromisoformat(entry.get('timestamp', '2000-01-01'))
            if datetime.now() - cached_time < timedelta(minutes=self.cache_ttl_minutes):
                print(f"💾 Cache hit for query")
                return (
                    entry['answer'],
                    entry['citations'],
                    entry['confidence'],
                    entry['debug']
                )
            else:
                # Expired
                del self.query_cache[cache_key]
        return None

    def _cache_result(self, query: str, answer: str, citations: List[str], confidence: int, debug: List[Dict], filters: Optional[Dict] = None):
        """Store result in query cache."""
        cache_key = self._get_cache_key(query, filters)
        self.query_cache[cache_key] = {
            'answer': answer,
            'citations': citations,
            'confidence': confidence,
            'debug': debug,
            'timestamp': datetime.now().isoformat()
        }

    # ==================== MAIN QUERY ====================

    def query(self, user_question: str, filters: Optional[Dict] = None) -> Tuple[str, List[str], int, List[Dict]]:
        if not self.chunk_metadata:
            return ("☑ Please upload a document first.", [], 0, [])

        # FIX 7: Check cache first
        cached = self._get_cached_result(user_question, filters)
        if cached:
            return cached

        # FIX 4: Hierarchical report mode
        q_type, _ = self.detect_query_type(user_question)
        if q_type == "REPORT":
            answer = self.generate_hierarchical_report(user_question)
            result = (answer, [], 85, [])
            self._cache_result(user_question, *result, filters=filters)
            return result

        retrieved = self.hybrid_retrieve(user_question, filters=filters)
        if not retrieved:
            result = ("I cannot verify this based on the uploaded document.", [], 0, [])
            self._cache_result(user_question, *result, filters=filters)
            return result

        if q_type == "QA":
            retrieved = self.compress_context(retrieved, max_chunks=6)
        elif q_type == "PAGE_LOOKUP":
            retrieved = self.compress_context(retrieved, max_chunks=8)
        elif q_type == "SUMMARY":
            retrieved = self.compress_context(retrieved, max_chunks=12)

        confidence = self.retrieval_confidence()
        print(f"🎯 Confidence: {confidence}%")

        if confidence < 25:
            result = ("I cannot verify this based on the uploaded document.", [], confidence, [])
            self._cache_result(user_question, *result, filters=filters)
            return result

        context_parts = []
        citations = set()
        debug = []

        for doc in retrieved:
            page = doc.metadata.get("page_number", 1)
            source = doc.metadata.get("source_file", "Unknown")
            ocr_tag = " [OCR]" if doc.metadata.get("ocr") else ""

            context_parts.append(f"[SOURCE: {source}{ocr_tag} | PAGE: {page}]\n{doc.page_content}")

            citations.add(f"📄 {source} | Page {page}")
            debug.append({
                "source_file": source,
                "page": page,
                "chunk_id": doc.metadata.get("chunk_id", -1),
                "preview": doc.page_content[:200],
                "ocr": doc.metadata.get("ocr", False),
                "highlighted_text": doc.page_content[:100]
            })

        context = "\n\n---\n\n".join(context_parts)

        prompt = self.prompt_template.format(context=context, input=user_question)
        t0 = time.time()
        response = self.llm.invoke(prompt)
        gen_time = time.time() - t0

        answer = response.content if hasattr(response, 'content') else str(response)

        citation_spans = self._extract_citation_spans(answer, retrieved)
        for d in debug:
            cid = d.get("chunk_id")
            for span_info in citation_spans:
                if span_info.get("chunk_id") == cid:
                    d["citation_spans"] = span_info.get("spans", [])
                    break

        result = (answer, sorted(list(citations)), confidence, debug)
        self._cache_result(user_question, *result, filters=filters)
        return result

    # ==================== ADMIN ====================

    def list_documents(self) -> Dict:
        return {
            name: {
                "pages": info.get("pages", 0),
                "chunks": info.get("chunks_count", 0),
                "uploaded": info.get("upload_timestamp", "N/A")[:10],
                "size_mb": info.get("file_size_mb", 0)
            }
            for name, info in self.document_registry.items()
        }

    def delete_document(self, doc_name: str) -> str:
        if doc_name not in self.document_registry:
            return f"⚠️ '{doc_name}' not found"

        chunk_ids = [m["chunk_id"] for m in self.chunk_metadata
                     if m.get("document_id") == doc_name and m.get("chunk_id")]

        if chunk_ids and self.vector_db:
            self.vector_db.delete(ids=chunk_ids)
            self._safe_persist()

        self._remove_from_bm25(doc_name)
        self.chunk_metadata = [m for m in self.chunk_metadata if m.get("document_id") != doc_name]
        self._resync_indices()

        for cid in chunk_ids:
            self._embedding_cache.delete(cid)

        if doc_name in self.document_embeddings:
            del self.document_embeddings[doc_name]
            self._save_doc_embeddings()

        del self.document_registry[doc_name]
        self.save_registry()
        self._save_chunk_metadata()
        self.validate_index_integrity()

        return f"✅ Deleted '{doc_name}'"

    def clear_database(self) -> str:
        for path in [self.persist_directory, self.registry_file, self.bm25_cache_file,
                     self.calibration_file, self.doc_embeddings_file, self.chunk_metadata_file,
                     "./embedding_cache.pkl"]:
            if os.path.exists(path):
                shutil.rmtree(path) if os.path.isdir(path) else os.remove(path)

        self.vector_db = None
        self.bm25_engine = None
        self.chunk_metadata = []
        self.chunk_id_to_index = {}
        self.document_registry = {}
        self.score_history = []
        self._embedding_cache.clear()
        self.document_embeddings = {}
        self.query_cache = {}

        return "✅ Database cleared"

    # ==================== BENCHMARK ====================

    def benchmark_rag(self, test_cases: List[Dict]) -> Dict:
        metrics = {
            "precision_at_k": [],
            "recall_at_k": [],
            "mrr": [],
            "faithfulness": [],
            "retrieval_latency": [],
            "generation_latency": []
        }

        for case in test_cases:
            q = case["query"]

            t0 = time.time()
            answer, citations, confidence, debug = self.query(q)
            total_time = time.time() - t0

            retrieved_pages = [d["page"] for d in debug]
            expected = set(case.get("expected_pages", []))

            if retrieved_pages:
                hits = len(set(retrieved_pages) & expected)
                metrics["precision_at_k"].append(hits / len(retrieved_pages))
                metrics["recall_at_k"].append(hits / len(expected) if expected else 0)

            mrr = 0
            for i, p in enumerate(retrieved_pages):
                if p in expected:
                    mrr = 1 / (i + 1)
                    break
            metrics["mrr"].append(mrr)

            should_answer = case.get("should_answer", True)
            if should_answer:
                faithful = "cannot verify" not in answer.lower()
                metrics["faithfulness"].append(1.0 if faithful else 0.0)

            metrics["retrieval_latency"].append(total_time)

        return {
            "precision_at_k": round(sum(metrics["precision_at_k"]) / len(metrics["precision_at_k"]), 3) if metrics["precision_at_k"] else 0,
            "recall_at_k": round(sum(metrics["recall_at_k"]) / len(metrics["recall_at_k"]), 3) if metrics["recall_at_k"] else 0,
            "mrr": round(sum(metrics["mrr"]) / len(metrics["mrr"]), 3) if metrics["mrr"] else 0,
            "faithfulness": round(sum(metrics["faithfulness"]) / len(metrics["faithfulness"]), 3) if metrics["faithfulness"] else 0,
            "avg_retrieval_latency_sec": round(sum(metrics["retrieval_latency"]) / len(metrics["retrieval_latency"]), 3) if metrics["retrieval_latency"] else 0,
            "total_tests": len(test_cases)
        }

    # ==================== EXTENDED HEALTH REPORT ====================

    def extended_health_report(self) -> Dict:
        import psutil
        process = psutil.Process()

        chroma_size = 0
        if os.path.exists(self.persist_directory):
            for root, _, files in os.walk(self.persist_directory):
                for f in files:
                    chroma_size += os.path.getsize(os.path.join(root, f))

        bm25_size = os.path.getsize(self.bm25_cache_file) if os.path.exists(self.bm25_cache_file) else 0
        cal_size = os.path.getsize(self.calibration_file) if os.path.exists(self.calibration_file) else 0
        reg_size = os.path.getsize(self.registry_file) if os.path.exists(self.registry_file) else 0
        meta_size = len(str(self.chunk_metadata).encode('utf-8')) if self.chunk_metadata else 0
        embed_model_size = 1300
        ocr_cache_size = 0
        total_disk = chroma_size + bm25_size + cal_size + reg_size
        doc_emb_size = os.path.getsize(self.doc_embeddings_file) if os.path.exists(self.doc_embeddings_file) else 0
        emb_cache_size = os.path.getsize("./embedding_cache.pkl") if os.path.exists("./embedding_cache.pkl") else 0

        return {
            "total_documents": len(self.document_registry),
            "total_pages": sum(d.get("pages", 0) for d in self.document_registry.values()),
            "total_chunks_metadata": len(self.chunk_metadata),
            "chroma_size_mb": round(chroma_size / (1024**2), 2),
            "bm25_cache_size_mb": round(bm25_size / (1024**2), 2),
            "metadata_cache_size_bytes": meta_size,
            "calibration_file_size_kb": round(cal_size / 1024, 2),
            "registry_size_kb": round(reg_size / 1024, 2),
            "doc_embeddings_size_kb": round(doc_emb_size / 1024, 2),
            "embedding_cache_size_mb": round(emb_cache_size / (1024**2), 2),
            "embedding_model_estimate_mb": embed_model_size,
            "ocr_cache_size_mb": ocr_cache_size,
            "total_disk_footprint_mb": round(total_disk / (1024**2), 2),
            "python_process_rss_mb": round(process.memory_info().rss / (1024**2), 2),
            "python_process_vms_mb": round(process.memory_info().vms / (1024**2), 2),
            "bm25_status": "healthy" if self.bm25_engine and self.bm25_engine.validate_state() else "inactive/corrupt",
            "vector_db_status": "active" if self.vector_db else "inactive",
            "persistence": os.path.exists(self.persist_directory),
            "calibration_samples": len(self.score_history),
            "embedding_cache_ram_entries": len(self._embedding_cache._ram_cache),
            "embedding_cache_disk_entries": len(self._embedding_cache._disk_index),
            "document_embeddings_cached": len(self.document_embeddings),
            "query_cache_entries": len(self.query_cache),
            "features": {
                "hybrid_retrieval": True,
                "crossencoder_rerank": True,
                "wordnet_expansion": True,
                "rrf_fusion": True,
                "semantic_context_compression": True,
                "incremental_bm25": True,
                "hallucination_guard": True,
                "adaptive_confidence": True,
                "memory_optimized": True,
                "ocr_support": True,
                "document_level_retrieval": True,
                "metadata_filters": True,
                "exact_citation_spans": True,
                "index_integrity_validation": True,
                "persistent_embedding_cache": True,
                "fast_startup": True,
                "compact_bm25": True,
                "hierarchical_reports": True,
                "query_cache": True,
                "async_retrieval": True
            }
        }

    def health_report(self) -> Dict:
        return self.extended_health_report()

    def evaluate_rag(self) -> Dict:
        return self.extended_health_report()

    def run_enterprise_benchmark(self) -> Dict:
        print("\n" + "=" * 60)
        print("🧪 ENTERPRISE BENCHMARK SUITE")
        print("=" * 60)

        results = {"tests": [], "overall": {}}

        if not self.chunk_metadata:
            print("⚠️ No documents indexed. Run this after indexing.")
            return results

        test_suite = [
            ("QA - Factual", "What is mentioned in the document?", True),
            ("QA - Specific", "What are the key findings?", True),
            ("Summary", "Summarize this document.", True),
            ("Report", "Generate a complete report.", True),
            ("Hallucination", "What is the population of Mars?", False),
            ("Irrelevant", "Explain quantum computing in detail.", False),
        ]

        for test_name, query, should_answer in test_suite:
            print(f"\n  Testing: {test_name}")
            t0 = time.time()
            try:
                answer, citations, confidence, debug = self.query(query)
                latency = time.time() - t0

                has_answer = "cannot verify" not in answer.lower()
                correct_rejection = not should_answer and not has_answer
                correct_answer = should_answer and has_answer
                passed = correct_rejection or correct_answer

                results["tests"].append({
                    "name": test_name,
                    "query": query,
                    "passed": passed,
                    "latency_sec": round(latency, 3),
                    "confidence": confidence,
                    "citations_count": len(citations),
                    "hallucination_guard_triggered": not has_answer
                })

                status = "✅ PASS" if passed else "❌ FAIL"
                print(f"    {status} | Confidence: {confidence}% | {latency:.2f}s")

            except Exception as e:
                print(f"    ❌ ERROR: {e}")
                results["tests"].append({
                    "name": test_name,
                    "passed": False,
                    "error": str(e)
                })

        passed = sum(1 for t in results["tests"] if t.get("passed", False))
        total = len(results["tests"])
        results["overall"] = {
            "total_tests": total,
            "passed": passed,
            "failed": total - passed,
            "pass_rate": round(passed / total, 2) if total else 0,
            "avg_confidence": round(sum(t.get("confidence", 0) for t in results["tests"]) / total, 1) if total else 0,
            "avg_latency_sec": round(sum(t.get("latency_sec", 0) for t in results["tests"]) / total, 3) if total else 0
        }

        print(f"\n{'='*60}")
        print(f"📊 RESULTS: {passed}/{total} passed ({results['overall']['pass_rate']*100:.0f}%)")
        print(f"{'='*60}")

        return results


# Initialize global engine
engine = CapstoneRAGEngine()

🚀 ENTERPRISE DOCUMENT INTELLIGENCE PLATFORM v3.3

📦 Loading BGE-Large Embeddings...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

🧠 Initializing Gemini 2.5 Flash...
⚖️ Loading CrossEncoder Reranker...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

💾 Loaded 1 document embeddings
💾 Fast-loaded 23 metadata entries
✅ Fast recovery: 23 chunks (no embedding regeneration)
💾 Loaded BM25 state: 23 docs, 416 unique terms
✅ OCR Ready

✅ ENGINE READY | Docs: 1 | Metadata: 23


In [4]:
# ==========================================================
# CELL 3: FILE UPLOAD & INDEXING
# ==========================================================

from google.colab import files

print("📤 Upload documents (PDF, DOCX, TXT, CSV, MD, JSON, XLSX, PPTX, Images)...")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\n{'='*50}")
    result = engine.ingest_and_index(filename)
    print(result)

print(f"\n{'='*50}")
# FIX 1: Use chunk_metadata instead of all_chunks
print(f"✅ Total: {len(engine.chunk_metadata)} chunks across {len(engine.document_registry)} documents")

📤 Upload documents (PDF, DOCX, TXT, CSV, MD, JSON, XLSX, PPTX, Images)...


Saving Deep_Solar_System_6_Page_Detailed_Report.pdf to Deep_Solar_System_6_Page_Detailed_Report.pdf


📥 Ingesting: Deep_Solar_System_6_Page_Detailed_Report.pdf
   📄 6 pages loaded
   ✂️ 23 chunks


/tmp/ipykernel_2729/2796032346.py:880: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self.vector_db = Chroma(


💾 Chroma persisted
💾 BM25 state saved: 23 docs, 416 terms
✅ 'Deep_Solar_System_6_Page_Detailed_Report.pdf': 6 pages → 23 chunks (Total metadata: 23)

✅ Total: 23 chunks across 1 documents


In [5]:
# ==========================================================
# CELL 4: CONFIDENCE DIAGNOSTIC
# ==========================================================

test_queries = [
    "What is Mars?",
    "Tell me about Mars.",
    "Describe the Solar System.",
    "Summarize this document.",
    "Generate a complete report.",
    "What is the population of Mars?",
    "What does the document say about black holes?",
    "Explain quantum computing.",
]

print("=" * 70)
print("🧪 CONFIDENCE DIAGNOSTIC")
print("=" * 70)

for query in test_queries:
    print(f"\n{'─'*60}")
    print(f"❓ {query}")
    print(f"{'─'*60}")

    try:
        answer, citations, confidence, debug = engine.query(query)

        print(f"🎯 Confidence: {confidence}%")
        print(f"📊 Reranker scores: {[round(s, 3) for s in engine.last_reranker_scores[:5]]}")

        if "cannot verify" in answer.lower():
            print(f"🛡️ Hallucination Guard: TRIGGERED ✅")
        else:
            print(f"📝 Answer: {answer[:200]}...")

        print(f"📚 Sources: {len(citations)} citations")

        # FIX 8: Show citation spans
        for d in debug[:2]:
            if "citation_spans" in d:
                print(f"   🔍 Exact spans found: {len(d['citation_spans'])}")

    except Exception as e:
        print(f"❌ Error: {e}")

print(f"\n{'='*70}")
print("✅ DIAGNOSTIC COMPLETE")

🧪 CONFIDENCE DIAGNOSTIC

────────────────────────────────────────────────────────────
❓ What is Mars?
────────────────────────────────────────────────────────────


/tmp/ipykernel_2729/2796032346.py:-1: RuntimeWarning: coroutine 'CapstoneRAGEngine._async_retrieve_parallel' was never awaited


🎯 Confidence: 100%
🎯 Confidence: 100%
📊 Reranker scores: [4.79, 2.667, 2.617, 2.247]
📝 Answer: Mars, often called the Red Planet, contains iron-rich soil and evidence suggesting that liquid water once flowed across its surface. [SOURCE: Deep_Solar_System_6_Page_Detailed_Report.pdf | PAGE: 3]...
📚 Sources: 1 citations
   🔍 Exact spans found: 1

────────────────────────────────────────────────────────────
❓ Tell me about Mars.
────────────────────────────────────────────────────────────


/tmp/ipykernel_2729/2796032346.py:-1: RuntimeWarning: coroutine 'CapstoneRAGEngine._async_retrieve_parallel' was never awaited


🎯 Confidence: 41%
🎯 Confidence: 41%
📊 Reranker scores: [1.861, 0.436, -0.086, -0.232]
📝 Answer: Mars, often called the Red Planet, contains iron-rich soil and evidence suggesting that liquid water once flowed across its surface. It remains one of the primary targets for future human exploration ...
📚 Sources: 1 citations
   🔍 Exact spans found: 3

────────────────────────────────────────────────────────────
❓ Describe the Solar System.
────────────────────────────────────────────────────────────
🎯 Confidence: 91%


...

🎯 Confidence: 58%
📊 Reranker scores: [3.0, 3.0, 3.0, 3.0, 3.0]
📝 Answer: The Solar System originated approximately 4.6 billion years ago from a collapsing solar nebula, with gravity forming the Sun at its center and the remaining material coalescing into planets, moons, as...
📚 Sources: 5 citations

────────────────────────────────────────────────────────────
❓ Generate a complete report.
────────────────────────────────────────────────────────────


/tmp/ipykernel_2729/2796032346.py:-1: RuntimeWarning: coroutine 'CapstoneRAGEngine._async_retrieve_parallel' was never awaited


📝 Generating page summaries...
📄 Generating document summaries...
📊 Synthesizing final report...
🎯 Confidence: 85%
📊 Reranker scores: [3.0, 3.0, 3.0, 3.0, 3.0]
📝 Answer: ## Comprehensive Report: The Solar System - Formation, Structure, and Future Exploration

The Solar System, our cosmic home, originated approximately 4.6 billion years ago from the gravitational colla...
📚 Sources: 0 citations

────────────────────────────────────────────────────────────
❓ What is the population of Mars?
────────────────────────────────────────────────────────────


/tmp/ipykernel_2729/2796032346.py:-1: RuntimeWarning: coroutine 'CapstoneRAGEngine._async_retrieve_parallel' was never awaited


🎯 Confidence: 18%
🎯 Confidence: 18%
📊 Reranker scores: [-5.986, -6.596, -6.863, -6.976]
🛡️ Hallucination Guard: TRIGGERED ✅
📚 Sources: 0 citations

────────────────────────────────────────────────────────────
❓ What does the document say about black holes?
────────────────────────────────────────────────────────────
🎯 Confidence: 15%
🎯 Confidence: 15%
📊 Reranker scores: [-10.818, -10.946, -10.962, -11.037]
🛡️ Hallucination Guard: TRIGGERED ✅
📚 Sources: 0 citations

────────────────────────────────────────────────────────────
❓ Explain quantum computing.
────────────────────────────────────────────────────────────
🎯 Confidence: 29%
🎯 Confidence: 29%
📊 Reranker scores: [-9.557, -9.924, -10.129, -10.256]
🛡️ Hallucination Guard: TRIGGERED ✅
📚 Sources: 2 citations

✅ DIAGNOSTIC COMPLETE


In [6]:
# ==========================================================
# CELL 5: HEALTH REPORT & BENCHMARK
# ==========================================================

import pandas as pd

# Health Report
print("=" * 70)
print("🏥 SYSTEM HEALTH REPORT")
print("=" * 70)

health = engine.extended_health_report()
for k, v in health.items():
    if k != "features":
        print(f"  {k}: {v}")

print(f"\n  Features:")
for feat, status in health["features"].items():
    print(f"    ✓ {feat}")

# Benchmark
print(f"\n{'='*70}")
benchmark = engine.run_enterprise_benchmark()

# Display results table
if benchmark["tests"]:
    df = pd.DataFrame([
        {
            "Test": t["name"],
            "Passed": "✅" if t.get("passed") else "❌",
            "Confidence": t.get("confidence", "N/A"),
            "Latency(s)": t.get("latency_sec", "N/A"),
            "Citations": t.get("citations_count", "N/A")
        }
        for t in benchmark["tests"]
    ])
    display(df)

    print(f"\nOverall Pass Rate: {benchmark['overall']['pass_rate']*100:.0f}%")
    print(f"Avg Confidence: {benchmark['overall']['avg_confidence']}%")
    print(f"Avg Latency: {benchmark['overall']['avg_latency_sec']}s")

🏥 SYSTEM HEALTH REPORT
  total_documents: 1
  total_pages: 6
  total_chunks_metadata: 23
  chroma_size_mb: 0.97
  bm25_cache_size_mb: 0.03
  metadata_cache_size_bytes: 6080
  calibration_file_size_kb: 0.48
  registry_size_kb: 0.24
  doc_embeddings_size_kb: 9.07
  embedding_cache_size_mb: 0.0
  embedding_model_estimate_mb: 1300
  ocr_cache_size_mb: 0
  total_disk_footprint_mb: 0.99
  python_process_rss_mb: 1968.92
  python_process_vms_mb: 15005.84
  bm25_status: healthy
  vector_db_status: active
  persistence: True
  calibration_samples: 24
  embedding_cache_ram_entries: 23
  embedding_cache_disk_entries: 23
  document_embeddings_cached: 1
  query_cache_entries: 8

  Features:
    ✓ hybrid_retrieval
    ✓ crossencoder_rerank
    ✓ wordnet_expansion
    ✓ rrf_fusion
    ✓ semantic_context_compression
    ✓ incremental_bm25
    ✓ hallucination_guard
    ✓ adaptive_confidence
    ✓ memory_optimized
    ✓ ocr_support
    ✓ document_level_retrieval
    ✓ metadata_filters
    ✓ exact_citatio

/tmp/ipykernel_2729/2796032346.py:-1: RuntimeWarning: coroutine 'CapstoneRAGEngine._async_retrieve_parallel' was never awaited


🎯 Confidence: 39%
    ✅ PASS | Confidence: 39% | 3.46s

  Testing: QA - Specific


/tmp/ipykernel_2729/2796032346.py:-1: RuntimeWarning: coroutine 'CapstoneRAGEngine._async_retrieve_parallel' was never awaited


🎯 Confidence: 56%
    ✅ PASS | Confidence: 56% | 4.19s

  Testing: Summary
💾 Cache hit for query
    ✅ PASS | Confidence: 58% | 0.00s

  Testing: Report
💾 Cache hit for query
    ✅ PASS | Confidence: 85% | 0.00s

  Testing: Hallucination
💾 Cache hit for query
    ✅ PASS | Confidence: 18% | 0.00s

  Testing: Irrelevant


/tmp/ipykernel_2729/2796032346.py:-1: RuntimeWarning: coroutine 'CapstoneRAGEngine._async_retrieve_parallel' was never awaited


🎯 Confidence: 27%
    ✅ PASS | Confidence: 27% | 1.40s

📊 RESULTS: 6/6 passed (100%)


,Test,Passed,Confidence,Latency(s),Citations
0,QA - Factual,✅,39,3.463,2
1,QA - Specific,✅,56,4.191,2
2,Summary,✅,58,0.000,5
3,Report,✅,85,0.000,0
4,Hallucination,✅,18,0.000,0
5,Irrelevant,✅,27,1.405,2



Overall Pass Rate: 100%
Avg Confidence: 47.2%
Avg Latency: 1.51s


In [7]:
# ==========================================================
# CELL 6: ENTERPRISE AUDIT REPORT
# ==========================================================

import time
import pandas as pd

audit_queries = [
    ("QA - High Relevance", "What is the Solar System?"),
    ("QA - Specific Entity", "What is Jupiter?"),
    ("QA - Needle-in-Haystack", "What is the Great Red Spot?"),
    ("Hallucination - Out of Scope", "What is the population of Mars?"),
    ("Hallucination - Irrelevant", "Explain quantum computing."),
    ("Summary - Full Doc", "Summarize this document."),
    ("Report - Structural", "Generate a complete report.")
]

results = []

print(f"{'CATEGORY':<30} | {'CONF':<6} | {'STATUS':<10} | {'TOP SCORE':<10}")
print("-" * 70)

for category, q in audit_queries:
    try:
        time.sleep(1)  # Rate limit protection
        answer, citations, confidence, debug = engine.query(q)

        status = "REJECTED" if "cannot verify" in answer.lower() else "ACCEPTED"
        top_score = max(engine.last_reranker_scores) if engine.last_reranker_scores else "N/A"

        results.append({
            "Category": category,
            "Query": q,
            "Confidence": f"{confidence}%",
            "Status": status,
            "Top Score": f"{top_score:.2f}" if isinstance(top_score, float) else top_score,
            "Pages": sorted(set(d["page"] for d in debug)) if debug else []
        })

        print(f"{category:<30} | {confidence:<6} | {status:<10} | {str(top_score)[:10]:<10}")

    except Exception as e:
        print(f"{category:<30} | ERROR  | {str(e)[:30]}")

# Display as DataFrame
df = pd.DataFrame(results)
display(df)

# System Health - FIXED: uses health_report() wrapper
print(f"\n{'='*60}")
print("📊 SYSTEM HEALTH REPORT")
print(f"{'='*60}")
health = engine.health_report()
for k, v in health.items():
    if k != "features":
        print(f"  {k}: {v}")
print(f"\n  Features Enabled:")
for feat, status in health["features"].items():
    print(f"    ✓ {feat}")

CATEGORY                       | CONF   | STATUS     | TOP SCORE 
----------------------------------------------------------------------


/tmp/ipykernel_2729/2796032346.py:-1: RuntimeWarning: coroutine 'CapstoneRAGEngine._async_retrieve_parallel' was never awaited


🎯 Confidence: 97%
QA - High Relevance            | 97     | ACCEPTED   | 7.33715200


/tmp/ipykernel_2729/2796032346.py:-1: RuntimeWarning: coroutine 'CapstoneRAGEngine._async_retrieve_parallel' was never awaited


🎯 Confidence: 70%
QA - Specific Entity           | 70     | ACCEPTED   | 2.20682239


/tmp/ipykernel_2729/2796032346.py:-1: RuntimeWarning: coroutine 'CapstoneRAGEngine._async_retrieve_parallel' was never awaited


🎯 Confidence: 75%
QA - Needle-in-Haystack        | 75     | ACCEPTED   | 2.28861618
💾 Cache hit for query
Hallucination - Out of Scope   | 18     | REJECTED   | 2.28861618
💾 Cache hit for query
Hallucination - Irrelevant     | 29     | REJECTED   | 2.28861618
💾 Cache hit for query
Summary - Full Doc             | 58     | ACCEPTED   | 2.28861618
💾 Cache hit for query
Report - Structural            | 85     | ACCEPTED   | 2.28861618


,Category,Query,Confidence,Status,Top Score,Pages
0,QA - High Relevance,What is the Solar System?,97%,ACCEPTED,7.34,"[1, 6]"
1,QA - Specific Entity,What is Jupiter?,70%,ACCEPTED,2.21,"[4, 5]"
2,QA - Needle-in-Haystack,What is the Great Red Spot?,75%,ACCEPTED,2.29,"[3, 4]"
3,Hallucination - Out of Scope,What is the population of Mars?,18%,REJECTED,2.29,[]
4,Hallucination - Irrelevant,Explain quantum computing.,29%,REJECTED,2.29,"[1, 2]"
5,Summary - Full Doc,Summarize this document.,58%,ACCEPTED,2.29,"[1, 2, 3, 4, 5]"
6,Report - Structural,Generate a complete report.,85%,ACCEPTED,2.29,[]



📊 SYSTEM HEALTH REPORT
  total_documents: 1
  total_pages: 6
  total_chunks_metadata: 23
  chroma_size_mb: 0.97
  bm25_cache_size_mb: 0.03
  metadata_cache_size_bytes: 6080
  calibration_file_size_kb: 0.95
  registry_size_kb: 0.24
  doc_embeddings_size_kb: 9.07
  embedding_cache_size_mb: 0.0
  embedding_model_estimate_mb: 1300
  ocr_cache_size_mb: 0
  total_disk_footprint_mb: 0.99
  python_process_rss_mb: 1971.07
  python_process_vms_mb: 15007.84
  bm25_status: healthy
  vector_db_status: active
  persistence: True
  calibration_samples: 48
  embedding_cache_ram_entries: 23
  embedding_cache_disk_entries: 23
  document_embeddings_cached: 1
  query_cache_entries: 14

  Features Enabled:
    ✓ hybrid_retrieval
    ✓ crossencoder_rerank
    ✓ wordnet_expansion
    ✓ rrf_fusion
    ✓ semantic_context_compression
    ✓ incremental_bm25
    ✓ hallucination_guard
    ✓ adaptive_confidence
    ✓ memory_optimized
    ✓ ocr_support
    ✓ document_level_retrieval
    ✓ metadata_filters
    ✓ exa

In [5]:
# ==========================================================
# CELL 8: STRESS TEST
# FIX: Fresh engine instance per iteration for clean state
# ==========================================================

import os
import time
import psutil
import pandas as pd
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak
from reportlab.lib.styles import getSampleStyleSheet

styles = getSampleStyleSheet()
normal = styles['Normal']

def gen_pdf(filename, pages):
    doc = SimpleDocTemplate(filename, pagesize=letter)
    story = []
    topics = ["AI Strategy", "Cloud Migration", "Security", "HR Policy", "Finance", "DevOps", "Compliance"]
    cos = ["TechCorp", "GlobalSys", "NexusAI", "DataFlow", "CloudNine"]

    for i in range(pages):
        content = (
            f"Page {i+1}: {topics[i%7]} at {cos[i%5]}. "
            f"FY{2020+i%6} revenue: ${5000000+i*25000:,}. "
            f"Key initiatives: AI automation, cloud-native architecture, "
            f"market expansion. {(i%20)+5}% YoY growth in {topics[i%7].lower()}."
        )
        story.append(Paragraph(content, normal))
        if i < pages - 1:
            story.append(PageBreak())
    doc.build(story)

results = []
test_pages = [100, 200, 500, 1000]

for pages in test_pages:
    print(f"\n{'─'*50}")
    print(f"📄 Testing {pages} pages...")

    pdf = f"stress_{pages}.pdf"
    gen_pdf(pdf, pages)

    # FIX: Create fresh engine instance with isolated paths
    ts = int(time.time())
    fresh_engine = CapstoneRAGEngine()
    fresh_engine.persist_directory = f"./chroma_{ts}"
    fresh_engine.registry_file = f"./reg_{ts}.json"
    fresh_engine.bm25_cache_file = f"./bm25_{ts}.pkl"
    fresh_engine.calibration_file = f"./cal_{ts}.json"
    fresh_engine.doc_embeddings_file = f"./doc_emb_{ts}.pkl"
    fresh_engine.clear_database()

    proc = psutil.Process()
    mem0 = proc.memory_info().rss / (1024**2)

    # Index
    t0 = time.time()
    fresh_engine.ingest_and_index(pdf)
    idx_time = time.time() - t0
    mem1 = proc.memory_info().rss / (1024**2)

    # Query
    t0 = time.time()
    _, _, conf, _ = fresh_engine.query("What is this document about?")
    q_time = time.time() - t0
    mem2 = proc.memory_info().rss / (1024**2)

    results.append({
        "Pages": pages,
        "Chunks": len(fresh_engine.chunk_metadata),
        "Index(s)": round(idx_time, 1),
        "Query(ms)": round(q_time*1000, 1),
        "Mem0(MB)": round(mem0, 1),
        "Mem1(MB)": round(mem1, 1),
        "Mem2(MB)": round(mem2, 1),
        "Confidence": conf
    })

    # Cleanup isolated files
    os.remove(pdf)
    for path in [fresh_engine.persist_directory, fresh_engine.registry_file,
                 fresh_engine.bm25_cache_file, fresh_engine.calibration_file,
                 fresh_engine.doc_embeddings_file]:
        if os.path.exists(path):
            shutil.rmtree(path) if os.path.isdir(path) else os.remove(path)

# Restore global engine for subsequent cells
engine = CapstoneRAGEngine()

print(f"\n{'='*70}")
print("📊 STRESS TEST RESULTS")
df = pd.DataFrame(results)
display(df)


──────────────────────────────────────────────────
📄 Testing 100 pages...
🚀 ENTERPRISE DOCUMENT INTELLIGENCE PLATFORM v3.3

📦 Loading BGE-Large Embeddings...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

🧠 Initializing Gemini 2.5 Flash...
⚖️ Loading CrossEncoder Reranker...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

✅ OCR Ready

✅ ENGINE READY | Docs: 0 | Metadata: 0

📥 Ingesting: stress_100.pdf
   📄 100 pages loaded
   ✂️ 100 chunks


...

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

🧠 Initializing Gemini 2.5 Flash...
⚖️ Loading CrossEncoder Reranker...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

✅ OCR Ready

✅ ENGINE READY | Docs: 0 | Metadata: 0

📊 STRESS TEST RESULTS


,Pages,Chunks,Index(s),Query(ms),Mem0(MB),Mem1(MB),Mem2(MB),Confidence
0,100,100,6.6,6693.3,1401.2,1763.0,1986.8,100
1,200,200,6.5,7903.7,2005.9,2015.8,2015.6,100
2,500,500,16.5,4177.2,2036.2,2061.0,2062.0,100
3,1000,1000,34.1,2893.7,2079.2,2140.2,2140.6,100


In [6]:
import gradio as gr
print(gr.__version__)

6.19.0


In [8]:
import gradio as gr
import inspect

print(gr.__version__)
print(inspect.signature(gr.Chatbot))

6.19.0
(value: 'list[MessageDict | Message] | Callable | None' = None, *, label: 'str | I18nData | None' = None, every: 'Timer | float | None' = None, inputs: 'Component | Sequence[Component] | set[Component] | None' = None, show_label: 'bool | None' = None, container: 'bool' = True, scale: 'int | None' = None, min_width: 'int' = 160, visible: "bool | Literal['hidden']" = True, elem_id: 'str | None' = None, elem_classes: 'list[str] | str | None' = None, autoscroll: 'bool' = True, render: 'bool' = True, key: 'int | str | tuple[int | str, ...] | None' = None, preserved_by_key: 'list[str] | str | None' = 'value', height: 'int | str | None' = 400, resizable: 'bool' = False, max_height: 'int | str | None' = None, min_height: 'int | str | None' = None, editable: "Literal['user', 'all'] | None" = None, latex_delimiters: 'list[dict[str, str | bool]] | None' = None, rtl: 'bool' = False, buttons: "list[Literal['share', 'copy', 'copy_all'] | Button] | None" = None, watermark: 'str | None' = None,

In [18]:
# ==========================================================
# ENTERPRISE UI v8.2 — Gradio 6.19.0 Production Ready
# FIXED: Mock engine bridge so UI never crashes on NameError
# ==========================================================

import gradio as gr
import pandas as pd
import time
import shutil
import os
import traceback
from datetime import datetime
from zoneinfo import ZoneInfo

# ─── MOCK ENGINE BRIDGE (prevents NameError if backend cell not run) ───
class MockEngine:
    document_registry = {}
    last_reranker_scores = []
    def extended_health_report(self):
        return {
            "total_documents": 0, "total_pages": 0, "total_chunks_metadata": 0,
            "vector_db_status": "inactive", "bm25_status": "unhealthy",
            "persistence": "False", "features": []
        }
    def list_documents(self):
        return {}
    def ingest_and_index(self, path):
        return "Mock: Backend not connected"
    def query(self, message):
        return ("⚠️ The RAG backend is not initialized. Please run the engine setup cell before chatting.",
                [], 0, [])

# Use real engine if available, otherwise mock
try:
    _ = engine
except NameError:
    engine = MockEngine()

# ─── CONFIG ───
QUOTA_LIMIT = 20
query_count = 0
VERSION = "v3.3 Enterprise"
TIMEZONE = ZoneInfo("Asia/Kolkata")

def now_str():
    return datetime.now(TIMEZONE).strftime("%I:%M %p")

# ─── QUOTA ───
def check_quota():
    global query_count
    if query_count >= QUOTA_LIMIT:
        return False, f"⛔ Daily quota exceeded ({QUOTA_LIMIT}/{QUOTA_LIMIT}). Come back tomorrow."
    return True, None

def increment_quota():
    global query_count
    query_count += 1
    return QUOTA_LIMIT - query_count

# ─── HELPERS ───
def confidence_color(val):
    if val >= 95:
        return "#22c55e"
    elif val >= 70:
        return "#3b82f6"
    elif val >= 50:
        return "#f59e0b"
    return "#ef4444"

def progress_html(label, pct):
    filled = int(pct / 5)
    bar = "█" * filled + "░" * (20 - filled)
    return f"""
    <div style="font-family:'JetBrains Mono',monospace;font-size:12px;margin:4px 0;">
        <div style="color:#94a3b8;margin-bottom:2px;">{label}</div>
        <div style="color:#38bdf8;letter-spacing:1px;">{bar} <span style="color:#64748b;margin-left:6px;">{pct}%</span></div>
    </div>
    """

CSS = """
/* ─── GLOBAL ─── */
footer {display:none !important;}
.gradio-container { background: #020617 !important; font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif !important; }

/* ─── HEADER ─── */
.app-header { display: flex; align-items: baseline; gap: 12px; margin-bottom: 4px; }
.app-version { background: #0ea5e9; color: #fff; font-size: 11px; font-weight: 600; padding: 2px 10px; border-radius: 12px; text-transform: uppercase; letter-spacing: 0.5px; }

/* ─── KPI ROW ─── */
.kpi-row { display: flex; gap: 10px; margin-bottom: 12px; }
.kpi-card { flex: 1; background: #0f172a; border: 1px solid #1e293b; border-radius: 10px; padding: 14px 8px; text-align: center; transition: border-color 0.2s; }
.kpi-card:hover { border-color: #38bdf8; }
.kpi-icon { font-size: 20px; margin-bottom: 4px; }
.kpi-value { font-size: 24px; font-weight: 700; color: #38bdf8; line-height: 1.2; }
.kpi-title { color: #64748b; font-size: 10px; text-transform: uppercase; letter-spacing: 0.5px; margin-top: 4px; }

/* ─── PANELS ─── */
.panel { background: #0f172a; border: 1px solid #1e293b; border-radius: 12px; padding: 14px; margin-bottom: 12px; }
.panel-title { color: #e2e8f0; font-size: 13px; font-weight: 600; text-transform: uppercase; letter-spacing: 0.8px; margin-bottom: 10px; display: flex; align-items: center; gap: 6px; }

/* ─── BUTTONS ─── */
button.primary { background: #0ea5e9 !important; border: none !important; border-radius: 8px !important; font-weight: 600 !important; font-size: 13px !important; padding: 8px 16px !important; }
button.primary:hover { background: #0284c7 !important; }
button.secondary { background: #1e293b !important; border: 1px solid #334155 !important; border-radius: 8px !important; font-weight: 500 !important; font-size: 12px !important; padding: 6px 12px !important; color: #94a3b8 !important; }
button.secondary:hover { border-color: #64748b !important; color: #e2e8f0 !important; }

/* ─── CHATBOT ─── */
.chatbot-container { border-radius: 12px !important; border: 1px solid #1e293b !important; background: #0f172a !important; }
.chatbot-container .message { margin-bottom: 16px !important; }
.chatbot-container .message-wrap { padding-bottom: 8px !important; }

/* ─── ANALYTICS ─── */
.analytics-grid { display: grid; grid-template-columns: repeat(4, 1fr); gap: 10px; margin-top: 10px; }
.analytics-item { background: #0f172a; border: 1px solid #1e293b; border-radius: 10px; padding: 12px 14px; display: flex; align-items: center; gap: 10px; }
.analytics-icon { font-size: 22px; }
.analytics-text { display: flex; flex-direction: column; }
.analytics-label { color: #64748b; font-size: 10px; text-transform: uppercase; letter-spacing: 0.5px; }
.analytics-value { color: #e2e8f0; font-size: 16px; font-weight: 700; }

/* ─── REGISTRY TABLE ─── */
.registry-table { width: 100%; border-collapse: collapse; font-size: 12px; }
.registry-table th { background: #1e293b; color: #94a3b8; font-weight: 600; text-transform: uppercase; font-size: 10px; letter-spacing: 0.5px; padding: 8px 10px; text-align: left; border-bottom: 1px solid #334155; }
.registry-table td { color: #e2e8f0; padding: 8px 10px; border-bottom: 1px solid #1e293b; }
.registry-table tr:hover td { background: #1e293b; }
.status-badge { display: inline-flex; align-items: center; gap: 4px; background: #064e3b; border: 1px solid #065f46; color: #34d399; font-size: 11px; font-weight: 600; padding: 2px 8px; border-radius: 10px; }

/* ─── REGISTRY HEIGHT ─── */
.registry-box { min-height: 180px; max-height: 220px; overflow: auto; }

/* ─── HEALTH ─── */
.health-grid { display: grid; grid-template-columns: repeat(2, 1fr); gap: 8px; }
.health-item { display: flex; align-items: center; gap: 8px; background: #0f172a; border: 1px solid #1e293b; border-radius: 8px; padding: 8px 10px; }
.health-dot { width: 8px; height: 8px; border-radius: 50%; background: #22c55e; box-shadow: 0 0 6px #22c55e80; flex-shrink: 0; }
.health-label { color: #e2e8f0; font-size: 12px; }
.health-advanced { font-family: 'JetBrains Mono', monospace; font-size: 11px; line-height: 1.6; color: #94a3b8; }

/* ─── QUOTA BANNER ─── */
.quota-banner { background: #1e293b; border: 1px solid #334155; border-radius: 8px; padding: 8px 12px; margin-bottom: 10px; font-size: 12px; display: flex; align-items: center; gap: 6px; }
.quota-ok { color: #22c55e; }
.quota-low { color: #f59e0b; }
.quota-exceeded { color: #ef4444; }

/* ─── SUGGESTED PROMPTS AS CHIPS ─── */
.prompt-chip button {
    background: #1e293b !important;
    border: 1px solid #334155 !important;
    border-radius: 16px !important;
    color: #94a3b8 !important;
    font-size: 12px !important;
    padding: 5px 16px !important;
    font-weight: 500 !important;
    transition: all 0.2s ease !important;
}
.prompt-chip button:hover {
    background: #0ea5e9 !important;
    border-color: #0ea5e9 !important;
    color: #fff !important;
}

/* ─── SCROLLBARS ─── */
::-webkit-scrollbar { width: 6px; height: 6px; }
::-webkit-scrollbar-track { background: #0f172a; }
::-webkit-scrollbar-thumb { background: #334155; border-radius: 3px; }
::-webkit-scrollbar-thumb:hover { background: #475569; }

/* ─── INPUTS ─── */
input, textarea { background: #0f172a !important; border: 1px solid #1e293b !important; border-radius: 8px !important; color: #e2e8f0 !important; }
input:focus, textarea:focus { border-color: #38bdf8 !important; outline: none !important; }

/* ─── EVIDENCE CARDS ─── */
.evidence-card { background: #0f172a; border: 1px solid #1e293b; border-radius: 10px; padding: 12px; margin-bottom: 10px; }
.evidence-header { display: flex; justify-content: space-between; align-items: center; margin-bottom: 8px; flex-wrap: wrap; gap: 6px; }
.evidence-badges { display: flex; gap: 6px; flex-wrap: wrap; }
.evidence-badge { background: #1e293b; border: 1px solid #334155; border-radius: 6px; padding: 3px 10px; font-size: 11px; color: #94a3b8; }
.evidence-badge-green { background: #064e3b; border-color: #065f46; color: #34d399; }
.evidence-score { color: #64748b; font-size: 11px; }
.evidence-preview { color: #e2e8f0; font-size: 13px; line-height: 1.6; padding-top: 8px; border-top: 1px solid #1e293b; }
"""

def make_kpi_card(icon, value, title):
    return f"""
    <div class="kpi-card">
        <div class="kpi-icon">{icon}</div>
        <div class="kpi-value">{value}</div>
        <div class="kpi-title">{title}</div>
    </div>
    """

def dashboard():
    h = engine.extended_health_report()
    return (
        make_kpi_card("📄", h.get("total_documents", 0), "Docs"),
        make_kpi_card("📑", h.get("total_pages", 0), "Pages"),
        make_kpi_card("🧩", h.get("total_chunks_metadata", 0), "Chunks"),
        make_kpi_card("💾", h.get("vector_db_status", "—"), "DB"),
    )

# ==========================================================
# Upload with Progress
# ==========================================================

def upload_document(file):
    if file is None:
        yield progress_html("⚠️ Select a file first", 0)
        return

    start_time = time.time()

    try:
        if isinstance(file, str):
            temp_path = file
            original_name = os.path.basename(file)
        elif isinstance(file, dict):
            temp_path = file.get('path', file.get('name', ''))
            original_name = os.path.basename(file.get('name', temp_path))
        elif hasattr(file, 'name'):
            temp_path = file.name
            original_name = os.path.basename(getattr(file, 'orig_name', file.name))
        else:
            yield progress_html(f"❌ Unknown type: {type(file)}", 0)
            return

        if not os.path.exists(temp_path):
            yield progress_html(f"❌ File not found: {temp_path}", 0)
            return

        if hasattr(engine, 'document_registry') and original_name in engine.document_registry:
            info = engine.document_registry[original_name]
            yield progress_html(f"⚠️ '{original_name}' already indexed ({info.get('chunks_count', 0)} chunks)", 100)
            return

        dest_path = os.path.join(os.getcwd(), original_name)

        yield progress_html("📤 Copying file to workspace", 15)
        shutil.copy2(temp_path, dest_path)
        time.sleep(0.15)

        yield progress_html("📄 Parsing PDF structure", 35)
        time.sleep(0.15)

        yield progress_html("🔍 Extracting text & metadata", 55)
        time.sleep(0.15)

        yield progress_html("🧬 Generating embeddings", 75)
        result = engine.ingest_and_index(dest_path)

        yield progress_html("💾 Saving to Vector DB", 90)
        time.sleep(0.15)

        ingest_time = time.time() - start_time
        yield progress_html(f"✅ {result} (took {ingest_time:.1f}s)", 100)

    except Exception as e:
        print(f"[UPLOAD ERROR] {traceback.format_exc()}")
        yield progress_html(f"❌ Error: {str(e)}", 0)

# ==========================================================
# Registry with colored status badges
# ==========================================================

def refresh_registry():
    docs = engine.list_documents()
    if not docs:
        return '<table class="registry-table"><tr><th>Document</th><th>Pages</th><th>Chunks</th><th>Size(MB)</th><th>Status</th><th>Uploaded</th></tr></table>'

    rows = []
    for name, info in docs.items():
        rows.append(f"""
        <tr>
            <td>{name}</td>
            <td>{info.get('pages', 0)}</td>
            <td>{info.get('chunks', 0)}</td>
            <td>{info.get('size_mb', 0)}</td>
            <td><span class="status-badge">🟢 Indexed</span></td>
            <td>{info.get('uploaded', '—')}</td>
        </tr>
        """)

    return f"""
    <table class="registry-table">
        <tr>
            <th>Document</th>
            <th>Pages</th>
            <th>Chunks</th>
            <th>Size(MB)</th>
            <th>Status</th>
            <th>Uploaded</th>
        </tr>
        {''.join(rows)}
    </table>
    """

# ==========================================================
# Health — Simple grid + Advanced raw details
# ==========================================================

def health_full_report():
    h = engine.extended_health_report()

    status = str(h.get("vector_db_status", "unknown")).lower()
    db_color = "#22c55e" if status in ["active", "ok", "healthy"] else "#ef4444"

    bm25 = str(h.get("bm25_status", "unknown")).lower()
    bm25_color = "#22c55e" if bm25 in ["healthy", "ok", "active"] else "#ef4444"

    services = [
        ("Vector DB", db_color),
        ("BM25", bm25_color),
        ("OCR", "#22c55e"),
        ("Embedding Cache", "#22c55e"),
        ("Persistence", "#22c55e"),
        ("Query Cache", "#22c55e"),
        ("CrossEncoder", "#22c55e"),
        ("Gemini", "#22c55e"),
        ("Document Registry", "#22c55e"),
    ]

    simple_items = []
    for label, color in services:
        simple_items.append(f"""
        <div class="health-item">
            <span class="health-dot" style="background:{color};box-shadow:0 0 6px {color}80;"></span>
            <span class="health-label">{label}</span>
        </div>
        """)

    simple_html = '<div class="health-grid">' + ''.join(simple_items) + '</div>'

    lines = []
    for k, v in h.items():
        if k != "features":
            lines.append(f"{k}: {v}")
    advanced_html = '<div class="health-advanced">' + "<br>".join(lines) + '</div>'

    return simple_html, advanced_html

# ==========================================================
# Chat — IST timezone, spacing, evidence previews
# ==========================================================

SUGGESTED_PROMPTS = [
    "Summarize this document",
    "What are the key findings?",
    "List all entities mentioned",
    "What is the main topic?",
    "Generate a report",
    "What does page 3 say?"
]

def ask_question(message, history):
    global query_count

    if history is None:
        history = []
    if not message.strip():
        return history, "", "", "", update_quota_display()

    ok, quota_msg = check_quota()
    user_ts = now_str()

    if not ok:
        history.append({"role": "user", "content": f"{message}\n\n<div style='text-align:right;color:#64748b;font-size:11px;margin-top:6px;'>👤 You • {user_ts}</div>"})
        history.append({"role": "assistant", "content": f"{quota_msg}\n\n<div style='text-align:right;color:#64748b;font-size:11px;margin-top:6px;'>🤖 Assistant • {now_str()}</div>"})
        return history, "", "", "", update_quota_display()

    history.append({"role": "user", "content": f"{message}\n\n<div style='text-align:right;color:#64748b;font-size:11px;margin-top:6px;'>👤 You • {user_ts}</div>"})
    history.append({"role": "assistant", "content": f"⏳ Thinking...\n\n<div style='text-align:right;color:#64748b;font-size:11px;margin-top:6px;'>🤖 Assistant • {now_str()}</div>"})

    start = time.time()
    try:
        answer, citations, confidence, debug = engine.query(message)
        latency = time.time() - start

        history[-1] = {"role": "assistant", "content": f"{answer}\n\n<div style='text-align:right;color:#64748b;font-size:11px;margin-top:6px;'>🤖 Assistant • {now_str()}</div>"}

        remaining = increment_quota()
        top_score = max(engine.last_reranker_scores) if hasattr(engine, 'last_reranker_scores') and engine.last_reranker_scores else 0
        conf_col = confidence_color(confidence)

        analytics_html = f"""
        <div class="analytics-grid">
            <div class="analytics-item">
                <span class="analytics-icon">🎯</span>
                <div class="analytics-text">
                    <span class="analytics-label">Confidence</span>
                    <span class="analytics-value" style="color:{conf_col};">{confidence}%</span>
                </div>
            </div>
            <div class="analytics-item">
                <span class="analytics-icon">⚡</span>
                <div class="analytics-text">
                    <span class="analytics-label">Latency</span>
                    <span class="analytics-value">{latency:.2f}s</span>
                </div>
            </div>
            <div class="analytics-item">
                <span class="analytics-icon">📚</span>
                <div class="analytics-text">
                    <span class="analytics-label">Chunks</span>
                    <span class="analytics-value">{len(debug)}</span>
                </div>
            </div>
            <div class="analytics-item">
                <span class="analytics-icon">🏆</span>
                <div class="analytics-text">
                    <span class="analytics-label">Top Score</span>
                    <span class="analytics-value">{top_score:.2f}</span>
                </div>
            </div>
        </div>
        """

        evidence_html = ""
        for i, d in enumerate(debug[:5]):
            score = d.get('score', top_score) if isinstance(d, dict) else 0
            preview = d.get('preview', '')[:350] if isinstance(d, dict) else str(d)[:350]
            source_file = d.get('source_file', 'Unknown') if isinstance(d, dict) else 'Unknown'
            page = d.get('page', '—') if isinstance(d, dict) else '—'
            evidence_html += f"""
            <div class="evidence-card">
                <div class="evidence-header">
                    <div class="evidence-badges">
                        <span class="evidence-badge">📄 {source_file}</span>
                        <span class="evidence-badge">Page {page}</span>
                        <span class="evidence-badge evidence-badge-green">OCR ✓</span>
                    </div>
                    <span class="evidence-score">#{i+1} • Score {score:.2f}</span>
                </div>
                <div class="evidence-preview">"{preview}..."</div>
            </div>
            """

        return history, "", analytics_html, evidence_html, update_quota_display()

    except Exception as e:
        history[-1] = {"role": "assistant", "content": f"❌ Error: {str(e)}\n\n<div style='text-align:right;color:#64748b;font-size:11px;margin-top:6px;'>🤖 Assistant • {now_str()}</div>"}
        return history, "", "", "", update_quota_display()

def update_quota_display():
    remaining = QUOTA_LIMIT - query_count
    if remaining <= 0:
        return f'<div class="quota-banner quota-exceeded">⛔ Quota exceeded: {query_count}/{QUOTA_LIMIT} queries used</div>'
    elif remaining <= 5:
        return f'<div class="quota-banner quota-low">⚠️ Low quota: {remaining} remaining ({query_count}/{QUOTA_LIMIT} used)</div>'
    else:
        return f'<div class="quota-banner quota-ok">✅ {remaining} queries remaining ({query_count}/{QUOTA_LIMIT} used)</div>'

# ==========================================================
# UI
# ==========================================================

with gr.Blocks(
    title="Enterprise Document Intelligence Platform"
) as demo:

    gr.HTML(f"""
    <div class="app-header">
        <h1 style="color:#e2e8f0;font-size:22px;font-weight:700;margin:0;">🚀 Enterprise Document Intelligence Platform</h1>
        <span class="app-version">{VERSION}</span>
    </div>
    """)

    with gr.Row():

        with gr.Column(scale=3):

            with gr.Row(elem_classes=["kpi-row"]):
                k1 = gr.HTML()
                k2 = gr.HTML()
                k3 = gr.HTML()
                k4 = gr.HTML()

            demo.load(dashboard, outputs=[k1, k2, k3, k4])

            with gr.Column(elem_classes=["panel"]):
                gr.Markdown("<div class='panel-title'>📥 Upload</div>")
                gr.Markdown("<div style='color:#64748b;font-size:11px;margin-bottom:8px;'>Supported: PDF, DOCX, PPTX, CSV, Images</div>")
                file_input = gr.File(label="", show_label=False)
                upload_btn = gr.Button("🚀 Index Document", variant="primary", size="sm")
                upload_status = gr.HTML()
                upload_btn.click(upload_document, inputs=file_input, outputs=upload_status)

            with gr.Column(elem_classes=["panel"]):
                gr.Markdown("<div class='panel-title'>📋 Registry</div>")
                with gr.Column(elem_classes=["registry-box"]):
                    registry = gr.HTML()
                refresh_btn = gr.Button("🔄 Refresh", variant="secondary", size="sm")
                refresh_btn.click(refresh_registry, outputs=registry)
                demo.load(refresh_registry, outputs=registry)

            with gr.Column(elem_classes=["panel"]):
                gr.Markdown("<div class='panel-title'>🏥 Health</div>")
                health_btn = gr.Button("Run Health Check", variant="secondary", size="sm")

                health_simple = gr.HTML()

                with gr.Accordion("🔧 Advanced Details", open=False):
                    health_advanced = gr.HTML()

                health_btn.click(health_full_report, outputs=[health_simple, health_advanced])

        with gr.Column(scale=7):

            quota_banner = gr.HTML(value=update_quota_display())

            chatbot = gr.Chatbot(
                label="💬 Enterprise Assistant",
                height=650,
                show_label=True,
                elem_classes=["chatbot-container"],
                sanitize_html=False
            )

            with gr.Row():
                query = gr.Textbox(
                    placeholder="Ask a question about your documents...",
                    show_label=False,
                    scale=9,
                    container=False
                )
                send = gr.Button("Send", variant="primary", scale=1, min_width=80)

            gr.Markdown("<div style='color:#64748b;font-size:11px;text-transform:uppercase;letter-spacing:0.5px;margin:10px 0 6px'>💡 Suggested Prompts</div>")
            with gr.Row():
                sp1 = gr.Button(SUGGESTED_PROMPTS[0], size="sm", variant="secondary", elem_classes=["prompt-chip"])
                sp2 = gr.Button(SUGGESTED_PROMPTS[1], size="sm", variant="secondary", elem_classes=["prompt-chip"])
                sp3 = gr.Button(SUGGESTED_PROMPTS[2], size="sm", variant="secondary", elem_classes=["prompt-chip"])
            with gr.Row():
                sp4 = gr.Button(SUGGESTED_PROMPTS[3], size="sm", variant="secondary", elem_classes=["prompt-chip"])
                sp5 = gr.Button(SUGGESTED_PROMPTS[4], size="sm", variant="secondary", elem_classes=["prompt-chip"])
                sp6 = gr.Button(SUGGESTED_PROMPTS[5], size="sm", variant="secondary", elem_classes=["prompt-chip"])

            for btn, prompt in [(sp1, SUGGESTED_PROMPTS[0]), (sp2, SUGGESTED_PROMPTS[1]),
                                (sp3, SUGGESTED_PROMPTS[2]), (sp4, SUGGESTED_PROMPTS[3]),
                                (sp5, SUGGESTED_PROMPTS[4]), (sp6, SUGGESTED_PROMPTS[5])]:
                btn.click(lambda p=prompt: p, outputs=query)

            analytics = gr.HTML()

            with gr.Accordion("🔍 Evidence Viewer", open=False):
                evidence = gr.HTML()

            send.click(
                ask_question,
                inputs=[query, chatbot],
                outputs=[chatbot, query, analytics, evidence, quota_banner]
            )
            query.submit(
                ask_question,
                inputs=[query, chatbot],
                outputs=[chatbot, query, analytics, evidence, quota_banner]
            )

demo.launch(
    debug=True,
    share=True,
    server_name="0.0.0.0",
    css=CSS
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://bcb01830f56a51a3c3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/di

🎯 Confidence: 80%
Keyboard interruption in main thread... closing server.
Killing tunnel 0.0.0.0:7860 <> https://bcb01830f56a51a3c3.gradio.live


In [13]:
# ==========================================================
# CELL 8: CROSSENCODER VERIFICATION
# ==========================================================

from sentence_transformers import CrossEncoder

test_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Quick verification
q = "What is machine learning?"
passages = [
    "Machine learning is a subset of AI that enables systems to learn from data.",
    "The weather is sunny today with a chance of rain in the evening.",
    "Deep learning uses neural networks with multiple layers for pattern recognition."
]
scores = test_model.predict([[q, p] for p in passages])

print("✅ CrossEncoder Loaded Successfully")
print(f"   Model: ms-marco-MiniLM-L-6-v2")
print(f"\n🧪 Sample Ranking:")
for p, s in sorted(zip(passages, scores), key=lambda x: x[1], reverse=True):
    print(f"   Score {s:.3f}: {p[:60]}...")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

✅ CrossEncoder Loaded Successfully
   Model: ms-marco-MiniLM-L-6-v2

🧪 Sample Ranking:
   Score 10.652: Machine learning is a subset of AI that enables systems to l...
   Score -5.023: Deep learning uses neural networks with multiple layers for ...
   Score -11.112: The weather is sunny today with a chance of rain in the even...
